In [ ]:
!git clone https://github.com/Seqaeon/EdgeTRM.git

In [ ]:
import os
os.chdir("/lambda/nfs/EdgeTRM")
print(os.getcwd())

!git config --global --add safe.directory /lambda/nfs/EdgeTRM
# !rm -rf /lambda/nfs/EdgeTRM/EdgeTRM
 


In [ ]:
!pwd

In [ ]:
# !git fetch origin
# !git reset --hard origin/main
!git pull origin main

In [ ]:
# ── MUST RUN FIRST — fix duplicate trm.py on Modal volume ────────────────────
# There are two identical trm.py files on the Modal volume:
#   TinyRecursiveModels/trm.py
#   TinyRecursiveModels/models/recursive_reasoning/trm.py
#
# Python loads them as separate module objects, so patching one class
# has no effect on instances created from the other.
#
# Fix: replace the top-level copy with a symlink so both import paths
# resolve to the same file and the same Python module object.
#
# Run this cell ONCE per kernel, then restart the kernel.

import os, sys

trm_root = None
for path in sys.path:
    candidate = os.path.join(path, "trm.py")
    if os.path.exists(candidate) and "TinyRecursiveModels" in candidate:
        trm_root = candidate
        break

# Also check the known Modal volume path directly
modal_top = "/lambda/nfs/EdgeTRM/TinyRecursiveModels/trm.py"
modal_sub = "/lambda/nfs/EdgeTRM/TinyRecursiveModels/models/recursive_reasoning/trm.py"

for top, sub in [(modal_top, modal_sub)]:
    if not os.path.exists(sub):
        print(f"[SKIP] {sub} not found")
        continue
    if os.path.islink(top):
        print(f"✓ Already a symlink: {top} → {os.readlink(top)}")
    elif os.path.exists(top):
        os.rename(top, top + ".bak")
        os.symlink(sub, top)
        print(f"✓ Replaced {top} with symlink → {sub}")
        print("  Restart the kernel now, then run all cells from the top.")
    else:
        print(f"[SKIP] {top} not found")


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
trm_root = repo_root / "TinyRecursiveModels"
if str(trm_root) not in sys.path:
    sys.path.insert(0, str(trm_root))

print("repo_root:", repo_root)
print("trm_root:", trm_root)

In [ ]:
# # 1. Install uv globally for the user session
# !pip install --user uv
# # 1. Create the virtual environment using uv (lightning fast)
# !uv venv .venv --

# # 2. Force copy mode for your local package on the NFS drive
# !uv pip install --link-mode=copy --python .venv/bin/python {trm_root}

# # 3. Force copy mode for einops
# !uv pip install --link-mode=copy --python .venv/bin/python einops


# !uv pip install --link-mode=copy --python .venv/bin/python matplotlib pandas

In [ ]:

!uv pip install {trm_root}
!uv pip install einops

In [ ]:
import sys

# Get the exact python binary your notebook is using right now
active_python = sys.executable
print(f"Targeting environment: {active_python}")

# Force uv to install into this exact environment path using copy mode
!uv pip install --link-mode=copy --python {active_python} matplotlib pandas einops

In [ ]:
# # Install ipykernel into your virtual environment
# !uv pip install --python .venv/bin/python ipykernel

# # Register the virtual environment as a new Jupyter Kernel
# ! .venv/bin/python -m ipykernel install --user --name=trm-env --display-name="Python (TRM Env)"


In [ ]:
!pip install pydantic

In [ ]:
# import os
# os.chdir('/lambda/nfs/EdgeTRM/TinyRecursiveModels')


In [ ]:
import torch
import yaml
from trm import TinyRecursiveReasoningModel_ACTV1

def load_arc_model(checkpoint_path, config_text):
    # 1. Parse the YAML
    raw_config = yaml.safe_load(config_text)
    
    # 2. Manually map and extract the required fields
    arch = raw_config['arch']
    
    # Construct the exact dictionary required by TinyRecursiveReasoningModel_ACTV1Config
    final_config = {
        "batch_size": 32,
        "seq_len": 81,
        "num_puzzle_identifiers": 1, # Sudoku has only 1 puzzle identifier
        "vocab_size": 11,             # Sudoku has vocab size 11
        "H_cycles": arch['H_cycles'],
        "L_cycles": arch['L_cycles'],
        "H_layers": arch['H_layers'],
        "L_layers": arch['L_layers'],
        "hidden_size": arch['hidden_size'],
        "expansion": arch['expansion'],
        "num_heads": arch['num_heads'],
        "pos_encodings": arch['pos_encodings'],
        "halt_max_steps": arch['halt_max_steps'],
        "halt_exploration_prob": arch['halt_exploration_prob'],
        "forward_dtype": arch.get('forward_dtype', 'bfloat16'),
        "mlp_t": arch.get('mlp_t', False),
        "puzzle_emb_ndim": arch.get('puzzle_emb_ndim', 512),
        "puzzle_emb_len": arch.get('puzzle_emb_len', 16),
        "no_ACT_continue": arch.get('no_ACT_continue', True)
    }

    # 1. Initialize the model
    model = TinyRecursiveReasoningModel_ACTV1(config_dict=final_config)
    
    # 2. Load the raw state_dict
    state_dict = torch.load(checkpoint_path, map_location='cpu')
    
    # 3. If the state_dict is nested under a 'model' key
    if 'model' in state_dict:
        state_dict = state_dict['model']
        
    # 4. Strip prefix and load cleaned state_dict
    unwanted_prefix = '_orig_mod.model.'
    clean_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith(unwanted_prefix):
            clean_state_dict[k[len(unwanted_prefix):]] = v
        else:
            clean_state_dict[k] = v
            
    # 5. Robust resizing of the puzzle embedding weights to preserve learned parameters
    puzzle_emb_name = "inner.puzzle_emb.weights"
    expected_shape = model.inner.puzzle_emb.weights.shape
    if puzzle_emb_name in clean_state_dict:
        puzzle_emb = clean_state_dict[puzzle_emb_name]
        if puzzle_emb.shape != expected_shape:
            print(f"Resizing puzzle embedding. Found {puzzle_emb.shape}, Expected {expected_shape}")
            new_weights = torch.empty(expected_shape, dtype=puzzle_emb.dtype, device=puzzle_emb.device)
            mean_emb = torch.mean(puzzle_emb, dim=0)
            new_weights[:] = mean_emb
            min_rows = min(puzzle_emb.shape[0], expected_shape[0])
            new_weights[:min_rows] = puzzle_emb[:min_rows]
            clean_state_dict[puzzle_emb_name] = new_weights
            
    # 6. Load the state dict
    model.load_state_dict(clean_state_dict)
    model.__dict__['model'] = model
    model.eval()
    print("Prefixes stripped and model loaded successfully!")
    return model

config_data = """
arch:
  H_cycles: 3
  H_layers: 0
  L_cycles: 6
  L_layers: 2
  expansion: 4
  forward_dtype: bfloat16
  halt_exploration_prob: 0.1
  halt_max_steps: 16
  hidden_size: 512
  num_heads: 8
  pos_encodings: none
  puzzle_emb_len: 16
  puzzle_emb_ndim: 512
  mlp_t: True
global_batch_size: 128
"""

checkpoint = "trm_sudoku_extreme/step_39060_sudoku_epoch_60k"

try:
    model = load_arc_model(checkpoint, config_data)
    print("Model successfully loaded!")
except Exception as e:
    print(f"Error: {e}")

model


In [ ]:
# ── Cell 9: Sparsity Audit ────────────────────────────────────────────────────
def count_zero_params(m):
    total, zeros = 0, 0
    for p in m.parameters():
        total += p.numel()
        zeros += (p == 0).sum().item()
    return zeros, total

for name, m in [('Original', model)]:
    z, t = count_zero_params(m)
    print(f'{name:<15} zero={z:,}/{t:,}  ({100*z/t:.1f}%)')

---
## Section 3 — High-Performance Test-Time Adaptation (TTA) & Evaluation Functions

This section defines the unified evaluation and training functions for our model compression experiments:
- **Function A: `evaluate_compressed_baseline`**: Fast, zero-shot evaluation using post-adaptation checkpoint embeddings.
- **Function B: `run_adaptation_eval`**: Evaluation of TTA convergence (training/adaptation from scratch).


In [ ]:
# ── 3.1  High-Performance Evaluation & Adaptation Helpers ─────────────────────
import os
import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

def get_inner(m):
    """Unwrap compiled/DDP model."""
    m2 = m.module if hasattr(m, 'module') else m
    return m2._orig_mod if hasattr(m2, '_orig_mod') else m2

@torch.no_grad()
def evaluate_arc_per_puzzle(mdl, loader, device="cpu", n_sup_max=16, max_batches=None, return_pass2=False, fast_mode=True, trunc_len=None):
    """
    Sudoku Extreme specific evaluation function.
    Compares the predicted path with target labels.
    """
    inner = get_inner(mdl)
    inner.eval()
    inner = inner.to(device)
    
    total_samples = 0
    total_correct_cells = 0
    total_cells = 0
    total_exact_correct = 0
    
    t0 = time.time()
    
    # We iterate over the dataloader batches
    batch_idx = 0
    for inputs, labels, pids in loader:
        if max_batches is not None and batch_idx >= max_batches:
            break
            
        inputs = inputs.to(device)
        labels = labels.to(device)
        pids = pids.to(device)
        
        if trunc_len is not None and trunc_len < inputs.shape[1]:
            inputs = inputs.clone()
            inputs[:, trunc_len:] = 0
            
        batch = {
            "inputs": inputs.to(torch.int32),
            "labels": labels.to(torch.int32),
            "puzzle_identifiers": pids.to(torch.int32),
        }
        
        carry = inner.initial_carry(batch)
        ic = carry.inner_carry
        cast = lambda t: t.to(device)
        
        from models.recursive_reasoning.trm import TinyRecursiveReasoningModel_ACTV1Carry, TinyRecursiveReasoningModel_ACTV1InnerCarry
        carry = TinyRecursiveReasoningModel_ACTV1Carry(
            inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(
                z_H=cast(ic.z_H), z_L=cast(ic.z_L)),
            steps=carry.steps.to(device),
            halted=carry.halted.to(device),
            current_data={k: v.to(device) for k, v in carry.current_data.items()},
        )
        
        for _ in range(n_sup_max):
            carry, outputs = inner(carry, batch)
            if carry.halted.all():
                break
                
        preds = torch.argmax(outputs["logits"], dim=-1)
        
        # In Sudoku Extreme, labels has shape (B, 81). 
        # The pad/ignore label ID in the dataset is 0. 
        # We ignore 0 tokens when computing cell accuracy.
        mask = (labels != 0)
        is_correct = mask & (preds == labels)
        
        correct_cells_batch = is_correct.sum().item()
        total_cells_batch = mask.sum().item()
        
        loss_counts = mask.sum(-1)
        seq_is_correct = (is_correct.sum(-1) == loss_counts) & (loss_counts > 0)
        exact_correct_batch = seq_is_correct.sum().item()
        
        total_correct_cells += correct_cells_batch
        total_cells += total_cells_batch
        total_exact_correct += exact_correct_batch
        total_samples += inputs.shape[0]
        
        batch_idx += 1
        
    elapsed = time.time() - t0
    
    cell_acc = total_correct_cells / total_cells if total_cells > 0 else 0.0
    pass_1_acc = total_exact_correct / total_samples if total_samples > 0 else 0.0
    pass_2_acc = pass_1_acc
    ms_per_puzzle = (elapsed / total_samples * 1000) if total_samples > 0 else 0.0
    
    if return_pass2:
        return pass_1_acc, pass_2_acc, cell_acc, ms_per_puzzle, total_samples
    else:
        return pass_1_acc, cell_acc, ms_per_puzzle, total_samples

def evaluate_arc(mdl, loader, device="cpu", n_sup_max=16, max_batches=None, return_pass2=False, fast_mode=True):
    res = evaluate_arc_per_puzzle(mdl, loader, device=device, n_sup_max=n_sup_max, max_batches=max_batches, return_pass2=return_pass2, fast_mode=fast_mode)
    if return_pass2:
        p1, p2, cell, ms, npuzz = res
        return p1, p2, cell, ms
    else:
        p1, cell, ms, npuzz = res
        return p1, cell, ms

def evaluate_compressed_baseline(model, checkpoint_path, data_dir, device="cuda", fast_mode=True):
    # 1. Clean and load the state dict from checkpoint_path
    state_dict = torch.load(checkpoint_path, map_location='cpu')
    if 'model' in state_dict:
        state_dict = state_dict['model']
        
    unwanted_prefix = '_orig_mod.model.'
    clean_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith(unwanted_prefix):
            clean_state_dict[k[len(unwanted_prefix):]] = v
        else:
            clean_state_dict[k] = v
            
    # 2. Resize model's puzzle embeddings if necessary
    inner_model = get_inner(model)
    puzzle_emb_name = "inner.puzzle_emb.weights"
    expected_shape = inner_model.puzzle_emb.weights.shape
    if puzzle_emb_name in clean_state_dict:
        puzzle_emb = clean_state_dict[puzzle_emb_name]
        if puzzle_emb.shape != expected_shape:
            print(f"[evaluate_compressed_baseline] Resizing puzzle embedding. Found {puzzle_emb.shape}, Expected {expected_shape}")
            new_weights = torch.empty(expected_shape, dtype=puzzle_emb.dtype, device=puzzle_emb.device)
            mean_emb = torch.mean(puzzle_emb, dim=0)
            new_weights[:] = mean_emb
            min_rows = min(puzzle_emb.shape[0], expected_shape[0])
            new_weights[:min_rows] = puzzle_emb[:min_rows]
            clean_state_dict[puzzle_emb_name] = new_weights
            
    inner_model.load_state_dict(clean_state_dict, strict=False)
    inner_model = inner_model.to(device)
    inner_model.eval()

    # 3. Create fresh SudokuDataset and DataLoader
    test_ds = SudokuDataset(f"{data_dir}/test")
    test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)

    print(f"[evaluate_compressed_baseline] Running direct baseline evaluation on {device} (fast_mode={fast_mode})...")
    p1, p2, cell, ms, npuzz = evaluate_arc_per_puzzle(
        inner_model, test_loader, device=device, n_sup_max=16, return_pass2=True, fast_mode=fast_mode
    )
    
    print(f"Evaluation Complete ({npuzz} puzzles):")
    print(f"  Pass@1 Accuracy: {p1*100:.2f}%")
    print(f"  Pass@2 Accuracy: {p2*100:.2f}%")
    print(f"  Cell Accuracy  : {cell*100:.2f}%")
    print(f"  Latency        : {ms:.2f} ms/puzzle")
    
    return p1, p2, cell, ms

# ── Function B: Train-Time Adaptation convergence loop from scratch ───────────
def run_adaptation_eval(model, data_dir, epochs=100, lr=1e-4, device="cuda", fast_mode=True):
    """
    Function B: Runs the training/TTA loop for a specified number of epochs on the passed model.
    """
    from trm import TinyRecursiveReasoningModel_ACTV1Carry, TinyRecursiveReasoningModel_ACTV1InnerCarry
    
    # 1. Freshly create SudokuDataset for train and test splits
    train_ds = SudokuDataset(f"{data_dir}/train")
    test_ds = SudokuDataset(f"{data_dir}/test")
    
    train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)
    
    inner_model = get_inner(model)
    inner_model = inner_model.to(device)
    
    # We set up a standard AdamW optimizer for simplicity and speed inside the notebook
    optimizer = torch.optim.AdamW(inner_model.parameters(), lr=lr)
    
    inner_model.train()
    print(f"[run_adaptation_eval] Starting adaptation training for {epochs} epochs on {device}...")
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        n_batches = 0
        
        for x_batch, y_true, pids in train_loader:
            x_batch = x_batch.to(device)
            y_true = y_true.to(device)
            pids = pids.to(device)
            
            batch = {
                "inputs": x_batch.to(torch.int32),
                "labels": y_true.to(torch.int32),
                "puzzle_identifiers": pids.to(torch.int32),
            }
            
            optimizer.zero_grad()
            
            # Initial carry
            carry = inner_model.initial_carry(batch)
            ic = carry.inner_carry
            cast = lambda t: t.to(device)
            carry = TinyRecursiveReasoningModel_ACTV1Carry(
                inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(
                    z_H=cast(ic.z_H), z_L=cast(ic.z_L)),
                steps=carry.steps.to(device),
                halted=carry.halted.to(device),
                current_data={k: v.to(device) for k, v in carry.current_data.items()},
            )
            
            # Forward pass over steps (similar to evaluate_arc_per_puzzle but with gradients)
            n_sup_max = inner_model.config.halt_max_steps if hasattr(inner_model.config, 'halt_max_steps') else 10
            
            for _ in range(n_sup_max):
                carry, outputs = inner_model(carry, batch)
                if carry.halted.all():
                    break
            
            loss = outputs["loss"]
            loss.backward()
            
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
            
        if (epoch + 1) % max(1, epochs // 10) == 0 or epoch == epochs - 1:
            print(f"  Epoch {epoch+1}/{epochs} - Avg Loss: {epoch_loss / max(1, n_batches):.4f}")
            
    # Evaluate at the end
    print("[run_adaptation_eval] Evaluating adapted model...")
    p1, p2, cell, ms, npuzz = evaluate_arc_per_puzzle(
        inner_model, test_loader, device=device, n_sup_max=16, return_pass2=True, fast_mode=fast_mode
    )
    
    print(f"Adaptation Complete ({npuzz} puzzles):")
    print(f"  Pass@1 Accuracy: {p1*100:.2f}%")
    print(f"  Pass@2 Accuracy: {p2*100:.2f}%")
    print(f"  Cell Accuracy  : {cell*100:.2f}%")
    
    return p1, p2, cell, ms

# Globally define TRMBackboneStep to prevent NameErrors downstream
class TRMBackboneStep(torch.nn.Module):
    """Single H-cycle backbone step. Puzzle embedding passed as float input."""
    def __init__(self, inner_model):
        super().__init__()
        self.m = inner_model

    def forward(self, x, puzzle_emb_row, z_H, z_L):
        from models.recursive_reasoning.trm import (
            TinyRecursiveReasoningModel_ACTV1Carry,
            TinyRecursiveReasoningModel_ACTV1InnerCarry,
        )
        # Patch puzzle_emb to return the pre-looked-up row
        orig = self.m.inner.puzzle_emb.forward
        cast_to = self.m.inner.puzzle_emb.cast_to
        def _injected(ids): return puzzle_emb_row.to(cast_to)
        self.m.inner.puzzle_emb.forward = _injected

        pids  = torch.zeros(x.shape[0], dtype=torch.int32, device=x.device)
        batch = {"inputs": x.to(torch.int32), "labels": x.to(torch.int32),
                 "puzzle_identifiers": pids}
        carry = TinyRecursiveReasoningModel_ACTV1Carry(
            inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(z_H=z_H, z_L=z_L),
            steps=torch.zeros(x.shape[0], dtype=torch.int32, device=x.device),
            halted=torch.zeros(x.shape[0], dtype=torch.bool, device=x.device),
            current_data=batch,
        )
        try:
            new_carry, outputs = self.m(carry, batch)
        finally:
            self.m.inner.puzzle_emb.forward = orig
        return outputs["logits"], new_carry.inner_carry.z_H, new_carry.inner_carry.z_L

# Globally define cast_carry_to_device to prevent device mismatch errors
def cast_carry_to_device(carry, device):
    from models.recursive_reasoning.trm import (
        TinyRecursiveReasoningModel_ACTV1Carry,
        TinyRecursiveReasoningModel_ACTV1InnerCarry,
    )
    ic = carry.inner_carry
    return TinyRecursiveReasoningModel_ACTV1Carry(
        inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(
            z_H=ic.z_H.to(device), z_L=ic.z_L.to(device)),
        steps=carry.steps.to(device),
        halted=carry.halted.to(device),
        current_data={k: v.to(device) for k, v in carry.current_data.items()},
    )


In [ ]:
# ── 4.0  Global imports & helpers ────────────────────────────────────────────
import sys, time, copy, json, math, warnings, io
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
warnings.filterwarnings("ignore")

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED     = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

def get_inner(m):
    """Unwrap DataParallel / compiled model to get inner TRM."""
    m2 = m.module if hasattr(m, 'module') else m
    return m2._orig_mod if hasattr(m2, '_orig_mod') else m2

print(f"Device: {DEVICE}")


In [ ]:
# ── 4.1  Sudoku Extreme DataLoader from pre-built .npy files ────────────────────────
import os
import csv
import json
import math
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

DATA_DIR = "./data/sudoku-extreme-full"

# Automatically download and build dataset if missing
if not os.path.exists(os.path.join(DATA_DIR, "test", "all__inputs.npy")):
    print("Dataset not found. Downloading and building Sudoku Extreme dataset from HuggingFace...")
    try:
        from huggingface_hub import hf_hub_download
        
        try:
            from tqdm import tqdm
        except Exception:
            def tqdm(iterable, *args, **kwargs):
                return iterable
        
        source_repo = "sapientinc/sudoku-extreme"
        
        def convert_subset(split_name: str, max_samples=1000):
            print(f"Downloading and processing '{split_name}' split...")
            inputs = []
            labels = []
            
            # Download from HuggingFace Hub
            csv_path = hf_hub_download(source_repo, f"{split_name}.csv", repo_type="dataset")
            
            with open(csv_path, newline="") as csvfile:
                reader = csv.reader(csvfile)
                next(reader)  # Skip header
                count = 0
                for row in reader:
                    # columns: source, q, a, rating
                    if len(row) < 3:
                        continue
                    q, a = row[1], row[2]
                    if max_samples is not None and count >= max_samples:
                        break
                    assert len(q) == 81 and len(a) == 81
                    q_clean = q.replace('.', '0')
                    inputs.append(np.frombuffer(q_clean.encode(), dtype=np.uint8).reshape(9, 9) - ord('0'))
                    labels.append(np.frombuffer(a.encode(), dtype=np.uint8).reshape(9, 9) - ord('0'))
                    count += 1
                    
            results = {k: [] for k in ["inputs", "labels", "puzzle_identifiers", "puzzle_indices", "group_indices"]}
            puzzle_id = 0
            example_id = 0
            results["puzzle_indices"].append(0)
            results["group_indices"].append(0)
            
            pbar = tqdm(zip(inputs, labels), total=len(inputs), desc=f"Converting {split_name}")
            for inp, out in pbar:
                results["inputs"].append(inp)
                results["labels"].append(out)
                example_id += 1
                puzzle_id += 1
                results["puzzle_indices"].append(example_id)
                results["puzzle_identifiers"].append(0)
                results["group_indices"].append(puzzle_id)
                
            def _seq_to_numpy(seq):
                arr = np.concatenate(seq).reshape(len(seq), -1)
                assert np.all((arr >= 0) & (arr <= 9))
                return arr + 1
                
            final_results = {
                "inputs": _seq_to_numpy(results["inputs"]),
                "labels": _seq_to_numpy(results["labels"]),
                "group_indices": np.array(results["group_indices"], dtype=np.int32),
                "puzzle_indices": np.array(results["puzzle_indices"], dtype=np.int32),
                "puzzle_identifiers": np.array(results["puzzle_identifiers"], dtype=np.int32),
            }
            
            metadata = {
                "seq_len": 81,
                "vocab_size": 11,
                "pad_id": 0,
                "ignore_label_id": 0,
                "blank_identifier_id": 0,
                "num_puzzle_identifiers": 1,
                "total_groups": len(final_results["group_indices"]) - 1,
                "mean_puzzle_examples": 1,
                "total_puzzles": len(final_results["group_indices"]) - 1,
                "sets": ["all"]
            }
            
            save_dir = os.path.join(DATA_DIR, split_name)
            os.makedirs(save_dir, exist_ok=True)
            
            with open(os.path.join(save_dir, "dataset.json"), "w") as f:
                json.dump(metadata, f)
                
            for k, v in final_results.items():
                np.save(os.path.join(save_dir, f"all__{k}.npy"), v)
                
        # Generate train and test splits (limit to 1000 for fast eval)
        os.makedirs(DATA_DIR, exist_ok=True)
        convert_subset("train", max_samples=1000)
        convert_subset("test", max_samples=1000)
        
        with open(os.path.join(DATA_DIR, "identifiers.json"), "w") as f:
            json.dump(["<blank>"], f)
            
        print("✓ Sudoku Extreme dataset successfully downloaded and generated in:", DATA_DIR)
    except Exception as e:
        print(f"Error generating dataset: {e}")

print(f"Using dataset directory: {DATA_DIR}")

class ARCDataset(Dataset):
    """
    Loads the pre-built Sudoku dataset from numpy arrays.
    """
    def __init__(self, split_dir: str):
        self.inputs = np.load(f"{split_dir}/all__inputs.npy")
        self.labels = np.load(f"{split_dir}/all__labels.npy")

        puzzle_ids  = np.load(f"{split_dir}/all__puzzle_identifiers.npy")  # (N_puzzles,)
        puzzle_ptr  = np.load(f"{split_dir}/all__puzzle_indices.npy")       # (N_puzzles+1,)

        # CSR expansion: each sample gets its puzzle's identifier
        counts = np.diff(puzzle_ptr).astype(np.int64)           # samples per puzzle
        self.per_sample_pids = np.repeat(puzzle_ids, counts)    # (N_samples,)

        assert len(self.inputs) == len(self.per_sample_pids), (
            f"Shape mismatch: inputs={len(self.inputs)}, pids={len(self.per_sample_pids)}"
        )

        with open(f"{split_dir}/../train/dataset.json") as fj:
            meta = json.load(fj)
        self.seq_len               = meta["seq_len"]
        self.vocab_size            = meta["vocab_size"]
        self.num_puzzle_identifiers = meta["num_puzzle_identifiers"]

    def __len__(self): return len(self.inputs)

    def __getitem__(self, i):
        return (
            torch.tensor(self.inputs[i],          dtype=torch.long),
            torch.tensor(self.labels[i],           dtype=torch.long),
            torch.tensor(self.per_sample_pids[i],  dtype=torch.long),
        )

SudokuDataset = ARCDataset

try:
    test_ds  = ARCDataset(f"{DATA_DIR}/test")
    train_ds = ARCDataset(f"{DATA_DIR}/train")
    BATCH_SIZE = 512
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    print(f"Test  split : {len(test_ds):,} samples  (seq_len={test_ds.seq_len})")
    print(f"Train split : {len(train_ds):,} samples")
    
    # Simple check on one batch to verify
    for x, y, pids in test_loader:
        print("Dataset loaded successfully!")
        break
except Exception as e:
    print(f"Error loading dataset: {e}")


### Note: `evaluate_arc` has been consolidated and moved to Section 3 for global accessibility.

In [ ]:
# ── 4.3  Baseline: evaluate the loaded FP32 model ────────────────────────────
# FP32 model runs on GPU (native bfloat16). INT8 dynamic-quant requires CPU.
if test_loader is not None and 'model' in dir():
    # Re-create dataloaders fresh from disk to clear any persistent in-memory clamped state
    test_ds = SudokuDataset(f"{DATA_DIR}/test")
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    if 'train_ds' in dir():
        train_ds = SudokuDataset(f"{DATA_DIR}/train")
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

    # Diagnostic Debug Prints to trace shape mismatch
    inner_model = get_inner(model)
    if hasattr(inner_model, 'puzzle_emb'):
        num_emb = inner_model.puzzle_emb.weights.shape[0]
        print(f"[DEBUG] model.puzzle_emb.weights shape: {inner_model.puzzle_emb.weights.shape}")
    else:
        num_emb = 30670
        print("[DEBUG] Model has no puzzle_emb module")
    
    print(f"[DEBUG] test_loader dataset max identifier: {test_loader.dataset.per_sample_pids.max()}")
    print(f"[DEBUG] test_loader dataset max identifier: {test_loader.dataset.per_sample_pids.max()}")
    
    # Safe index correction: clamp out-of-bounds indices to prevent CUDA assertion failures

    # Assert indices are safe in our resized lookup table
    assert test_loader.dataset.per_sample_pids.max() < num_emb, \
        f"Out-of-bounds indices! Max pid {test_loader.dataset.per_sample_pids.max()} >= num_emb {num_emb}"
    print("[DEBUG] Dataset indices verified safe. No clamping needed!")
    print(f"Running FP32 baseline evaluation on {DEVICE}...")
    fp32_p1, fp32_p2, fp32_cell, fp32_ms = evaluate_arc(
        model, test_loader, device=str(DEVICE), n_sup_max=10, max_batches=None, return_pass2=True
    )
    print(f"\nFP32 Baseline (All test puzzles):")
    print(f"  Pass@1 Exact: {fp32_p1:.4f}")
    print(f"  Pass@2 Exact: {fp32_p2:.4f}")
    print(f"  Cell Acc    : {fp32_cell:.4f}")
    print(f"  Latency     : {fp32_ms*1000:.2f} ms/puzzle")
else:
    print("[SKIP] model or test_loader not available.")

---
## Section 4 — Quantization Wrappers

| Level | Method | API | Expected size reduction |
|---|---|---|---|
| FP32 | Baseline | — | 1× |
| INT8 | Dynamic quantization | `torch.quantization.quantize_dynamic` | ~4× |
| INT4 | Fake quantization | Custom `FakeQuantINT4` | ~8× (simulated) |


In [ ]:
# ── 4.4  Quantization helpers ─────────────────────────────────────────────────
from models.layers import CastedLinear   # NOT a subclass of nn.Linear — needs explicit handling

# ── A. FP16 (half-precision) — GPU, same memory as BF16 ──────────────────────
def quantize_fp16(mdl: nn.Module) -> nn.Module:
    """Cast model to float16. Runs on GPU. Same param count as BF16 (2 bytes/param),
    useful for checking numerical sensitivity between BF16 and FP16."""
    m = copy.deepcopy(get_inner(mdl)).cuda().half()
    return m

# ── B. INT8 via bitsandbytes — GPU, ~2× smaller backbone ─────────────────────
def quantize_int8_bnb(mdl: nn.Module) -> nn.Module:
    """
    Replace CastedLinear + nn.Linear layers with bitsandbytes Linear8bitLt.
    Weights are stored as INT8 (quantized lazily on first forward pass on CUDA).
    ~2× smaller backbone vs FP16; real GPU INT8 kernels (not fake quant).
    Excludes lm_head and q_head as their small output dims (not multiples of 8) cause cublasLt errors.
    """
    try:
        import bitsandbytes as bnb
        from bitsandbytes.nn import Linear8bitLt
    except ImportError:
        print("[ERROR] bitsandbytes not installed. Run: pip install bitsandbytes")
        return None

    m = copy.deepcopy(get_inner(mdl)).cuda()
    replaced = 0

    for name, module in list(m.named_modules()):
        if not isinstance(module, (nn.Linear, CastedLinear)):
            continue
        # Exclude lm_head and q_head to avoid cublasLt shape errors on non-multiples of 8
        if "lm_head" in name or "q_head" in name:
            continue

        out_f, in_f = module.weight.shape
        has_bias = module.bias is not None

        new_layer = Linear8bitLt(
            in_f, out_f,
            bias=has_bias,
            has_fp16_weights=False,  # store INT8 persistently (not FP16 + dequant)
            threshold=6.0,           # LLM.int8() outlier threshold
        ).cuda()
        # Copy weight as Int8Params — quantized on first forward
        new_layer.weight = bnb.nn.Int8Params(
            module.weight.data.to(torch.float16),
            requires_grad=False,
            has_fp16_weights=False,
        )
        if has_bias:
            new_layer.bias = nn.Parameter(module.bias.data.to(torch.float16))

        parts = name.split(".")
        parent = m
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1], new_layer)
        replaced += 1

    print(f"  Replaced {replaced} CastedLinear/Linear → bnb.Linear8bitLt (GPU INT8)")
    return m

# ── C. INT8 PyTorch dynamic (CPU only, legacy fallback) ──────────────────────
def quantize_int8_cpu(mdl: nn.Module) -> nn.Module:
    """PyTorch dynamic INT8 — CPU only. Slow but works without bitsandbytes."""
    q = copy.deepcopy(get_inner(mdl)).cpu().float()
    q = torch.quantization.quantize_dynamic(q, {nn.Linear}, dtype=torch.qint8)
    return q

# ── D. INT4 fake quantization (GPU) ──────────────────────────────────────────
class FakeQuantINT4(nn.Module):
    """Symmetric per-tensor fake INT4 quant wrapper — works on CastedLinear too."""
    def __init__(self, weight: torch.Tensor, bias=None):
        super().__init__()
        self.weight = nn.Parameter(weight.clone(), requires_grad=False)
        self.bias   = nn.Parameter(bias.clone(), requires_grad=False) if bias is not None else None

    def _fake_quant(self, x: torch.Tensor, n_bits: int = 4) -> torch.Tensor:
        q_max = 2 ** (n_bits - 1) - 1
        scale = x.float().abs().max().clamp(min=1e-8) / q_max
        x_q   = torch.clamp((x.float() / scale).round(), -q_max, q_max)
        return (x_q * scale).to(x.dtype)

    def forward(self, x):
        # Cast fake-quant weight+bias to input dtype (e.g. bfloat16 on GPU)
        w_q = self._fake_quant(self.weight).to(x.dtype)
        b   = self.bias.to(x.dtype) if self.bias is not None else None
        return F.linear(x, w_q, b)

def quantize_int4_fake(mdl: nn.Module) -> nn.Module:
    """Replace CastedLinear + nn.Linear with fake INT4 wrappers. GPU-compatible."""
    m = copy.deepcopy(get_inner(mdl))
    replaced = 0
    for name, module in list(m.named_modules()):
        if not isinstance(module, (nn.Linear, CastedLinear)):
            continue
        parts  = name.split(".")
        parent = m
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1],
                FakeQuantINT4(module.weight, module.bias))
        replaced += 1
    print(f"  Replaced {replaced} CastedLinear/Linear → FakeQuantINT4")
    return m

# ── E. Size estimators ────────────────────────────────────────────────────────
def estimate_size_kb(mdl: nn.Module, bits: int = 32) -> float:
    """Estimate backbone (params only) storage at given bit-width."""
    n = sum(p.numel() for p in get_inner(mdl).parameters())
    return n * bits / 8 / 1024

def actual_size_kb(mdl: nn.Module) -> float:
    buf = io.BytesIO()
    torch.save(get_inner(mdl).state_dict(), buf)
    return buf.tell() / 1024



In [ ]:
!uv pip install bitsandbytes

In [ ]:
# ── 4.7  Instantiate all model variants ──────────────────────────────────────
# Device map:
#   FP16       → GPU  (float16, same memory as BF16)
#   BnB INT8   → GPU  (real INT8 via bitsandbytes LLM.int8())
#   INT4 fake  → GPU  (simulated INT4, float32 ops)
#   CPU INT8   → CPU  (PyTorch dynamic — slow, legacy comparison)

variants = {}   # name → (model, device)

if 'model' in dir():
    inner = get_inner(model)

    print("Creating FP16 variant...")
    variants["FP16"]        = (quantize_fp16(inner), "cuda")

    print("Creating BnB INT8 variant (GPU)...")
    bnb_model = quantize_int8_bnb(inner)
    if bnb_model is not None:
        variants["INT8 (bnb)"]  = (bnb_model, "cuda")

    print("Creating INT4 fake-quant variant...")
    int4_model = quantize_int4_fake(inner)
    if torch.cuda.is_available():
        int4_model = int4_model.cuda()
    variants["INT4 (fake)"] = (int4_model, "cuda" if torch.cuda.is_available() else "cpu")

    # Keep FP32 (original, on GPU/BF16) as baseline
    variants["FP32 (bf16)"] = (inner, str(DEVICE))

    # Optional: CPU INT8 legacy (slow — comment out if not needed)
    # print("Creating CPU INT8 variant (slow)...")
    # variants["INT8 (cpu)"] = (quantize_int8_cpu(inner), "cpu")

    print("\nModel Variant    | Device | Backbone KB (params only)")
    print("-" * 52)
    bits_map = {"FP32 (bf16)": 16, "FP16": 16, "INT8 (bnb)": 8, "INT4 (fake)": 4, "INT8 (cpu)": 8}
    for vname, (vm, dev) in variants.items():
        kb = estimate_size_kb(vm, bits_map.get(vname, 32))
        print(f"  {vname:<16} | {dev:<6} | {kb:>10.1f} KB")
else:
    print("[SKIP] model not loaded.")
    variants = {}


---
## Section 5 — Reasoning Decay Analysis

**Hypothesis:** Quantization degrades Sudoku reasoning in a predictable, architecture-dependent way.

We compare FP32 → INT8 → INT4 across:
- **Exact match** (entire grid correct)
- **Cell accuracy** (per-token correctness)
- **Inference latency** (ms/puzzle)


In [ ]:
# ── 5.1  Evaluate all quantization levels ─────────────────────────────────────
if test_loader is not None and variants:
    print("Evaluating all variants on Sudoku test set (All batches)...\n")
    results = {}
    for vname, (vm, dev) in variants.items():
        print(f"  Evaluating {vname} on {dev}...")
        try:
            # exact1, exact2, cell, ms_pp = evaluate_arc(vm, test_loader, device=dev, n_sup_max=10, max_batches=None, return_pass2=True)
            exact1, exact2, cell, ms_pp  = evaluate_arc(vm, test_loader, device=dev, n_sup_max=10, max_batches=None, return_pass2=True)

            results[vname] = {"exact1": exact1, "exact2": exact2, "exact_acc": exact1, "cell_acc": cell, "latency_ms": ms_pp * 1000}
            print(f"  {vname:<16} | Pass@1={exact1:.4f} | Pass@2={exact2:.4f} | cell={cell:.4f} | {ms_pp*1000:.2f} ms/puzzle")
        except Exception as e:
            print(f"  [WARN] {vname} failed: {e}")

    print("\n── Reasoning Decay (drop from FP32 baseline) ─────────────────────")
    fp32_e = results.get("FP32 (bf16)", {}).get("exact1", 0)
    fp32_c = results.get("FP32 (bf16)", {}).get("cell_acc",  0)
    for vname, r in results.items():
        if vname == "FP32 (bf16)": continue
        de = fp32_e - r["exact1"]
        dc = fp32_c - r["cell_acc"]
        mode = "CATASTROPHIC" if de > 0.5 else ("GRACEFUL" if de < 0.2 else "MODERATE")
        print(f"  {vname:<16} | ΔPass@1={de:+.4f} | Δcell={dc:+.4f} | {mode}")
else:
    results = {}
    print("[SKIP] variants or test_loader not available.")


In [ ]:
print(torch.__version__)

In [ ]:
fp32_e = results.get("FP32 (bf16)", {}).get("exact2", 0)

for vname, r in results.items():
    if vname == "FP32 (bf16)": continue
    de = fp32_e - r["exact2"]
    dc = fp32_c - r["cell_acc"]
    mode = "CATASTROPHIC" if de > 0.5 else ("GRACEFUL" if de < 0.2 else "MODERATE")
    print(f"  {vname:<16} | ΔPass@2={de:+.4f} | Δcell={dc:+.4f} | {mode}")

In [ ]:
results

In [ ]:
# ── 5.2  Visualise reasoning decay ────────────────────────────────────────────
if results:
    names      = list(results.keys())
    exact_accs = [results[n]["exact1"]  for n in names]
    cell_accs  = [results[n]["cell_acc"]   for n in names]
    latencies  = [results[n]["latency_ms"] for n in names]

    palette = ["#2196F3", "#00BCD4", "#4CAF50", "#FF9800", "#FF5722"]
    colors  = palette[:len(names)]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Sudoku Extreme Reasoning Decay Under Quantization", fontsize=14, fontweight="bold")

    # Plot 1: exact match
    axes[0].bar(names, exact_accs, color=colors, alpha=0.85, edgecolor="black", lw=0.8)
    axes[0].set_title("Exact Match Accuracy", fontsize=12, fontweight="bold")
    axes[0].set_ylabel("Accuracy"); axes[0].set_ylim(0, 1)
    axes[0].tick_params(axis="x", rotation=20)
    for i, v in enumerate(exact_accs):
        axes[0].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)

    # Plot 2: exact vs cell
    x = np.arange(len(names)); w = 0.35
    axes[1].bar(x - w/2, exact_accs, w, label="Exact Match", color=colors, alpha=0.8, edgecolor="black")
    axes[1].bar(x + w/2, cell_accs,  w, label="Cell Acc",    color=colors, alpha=0.4, edgecolor="black", hatch="//")
    axes[1].set_title("Graceful vs Catastrophic Decay", fontsize=12, fontweight="bold")
    axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=20)
    axes[1].set_ylabel("Accuracy"); axes[1].set_ylim(0, 1.1); axes[1].legend()

    # Plot 3: accuracy vs latency
    for i, (n, e, lat) in enumerate(zip(names, exact_accs, latencies)):
        axes[2].scatter(lat, e, color=colors[i], s=180, zorder=5)
        axes[2].annotate(n, (lat, e), textcoords="offset points", xytext=(6, 4), fontsize=8)
    axes[2].set_title("Accuracy–Latency Frontier", fontsize=12, fontweight="bold")
    axes[2].set_xlabel("Latency (ms/puzzle)"); axes[2].set_ylabel("Exact Match Accuracy")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("reasoning_decay_arc.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: reasoning_decay_arc.png")


---
## Section 6 — Recursive Depth × Quantization Grid

**Question:** Does more recursion depth (H_cycles) buffer against quantization noise?

We sweep `H_cycles ∈ {1, 2, 3, 4}` by temporarily overriding the config at inference time
(the model was trained with H_cycles=3, so values above that are out-of-distribution).


In [ ]:
# ── 6.1  Recursive depth sweep at inference time ──────────────────────────────
def evaluate_at_depth(mdl, loader, h_cycles_override: int, device="cpu", max_batches=None, n_sup_max=1):
    """Temporarily override H_cycles for a depth-sweep evaluation."""
    inner = get_inner(mdl)
    orig_h  = inner.config.H_cycles
    orig_ih = inner.inner.config.H_cycles
    try:
        inner.config.H_cycles = h_cycles_override
        inner.inner.config.H_cycles = h_cycles_override
        # Use n_sup_max=n_sup_max so we can sweep cycles at a specific outer step count
        exact1, exact2, cell, ms_per_puzzle = evaluate_arc(
            inner, loader, device=device, n_sup_max=n_sup_max, max_batches=max_batches, return_pass2=True
        )
    finally:
        inner.config.H_cycles = orig_h
        inner.inner.config.H_cycles = orig_ih
    return exact1, exact2, cell, ms_per_puzzle

H_CYCLES_SWEEP = [1, 2, 3, 4]   # trained at H=3
N_SUP_MAX_SWEEP = [1, 2, 4, 6, 8, 10]
grid_results   = {}

if test_loader is not None and variants:
    print(f"{'Quant':<14} {'H':<3} {'n_sup':<5} {'Pass@1':>9} {'Pass@2':>9} {'Cell Acc':>9} {'Latency':>10} {'GFLOPs':>9}")
    print("-" * 75)
    for qname, (qmodel, dev) in variants.items():
        for hc in H_CYCLES_SWEEP:
            for n_sup in N_SUP_MAX_SWEEP:
                try:
                    exact1, exact2, cell, ms_pp = evaluate_at_depth(
                        qmodel, test_loader, hc, device=dev, max_batches=None, n_sup_max=n_sup
                    )
                    exact = exact1
                except Exception as e:
                    print(f"  [WARN] {qname} H={hc} n_sup={n_sup}: {e}")
                    exact1, exact2, cell, ms_pp = 0.0, 0.0, 0.0, 0.0
                
                # GFLOPs: Each H-cycle in a step uses ~62.5 GFLOPs based on Section 17.3 profiling
                flops = n_sup * hc * (187504.0 / 3.0) / 1000.0
                grid_results[(qname, hc, n_sup)] = (exact1, exact2, cell, ms_pp, flops)
                print(f"{qname:<14} {hc:<3d} {n_sup:<5d} {exact1:>9.4f} {exact2:>9.4f} {cell:>9.4f} {ms_pp:>10.2f} {flops:>9.1f}")
else:
    print("[SKIP] variants not available.")


In [ ]:
# ── 6.2  Plot depth × quantization grid ─────────────────────────────────────
if 'grid_results' in dir() and grid_results:
    import matplotlib.pyplot as plt

    quant_levels = list(dict.fromkeys(k[0] for k in grid_results))
    # Support clean color mapping for any model variants present
    colors_map = {
        "FP32 (bf16)": "#2196F3",
        "FP16":        "#9C27B0",
        "INT8 (bnb)":  "#4CAF50",
        "INT4 (fake)": "#FF5722"
    }
    marker_map = {
        "FP32 (bf16)": "o",
        "FP16":        "D",
        "INT8 (bnb)":  "s",
        "INT4 (fake)": "^"
    }

    def plot_frontier(x_metric, x_label, title_suffix, filename):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
        
        for qname in quant_levels:
            pts = []
            for (qn, hc, n_sup), (ex1, ex2, cell_acc, ms, flops) in grid_results.items():
                if qn == qname:
                    x_val = flops if x_metric == "flops" else ms
                    pts.append((x_val, ex1, cell_acc))
            
            if not pts:
                continue
            
            c = colors_map.get(qname, "gray")
            m = marker_map.get(qname, "o")
            
            # Pareto frontier for exact1 (Pass@1)
            # Sort by x ascending, then by y descending
            pts_sorted_ex = sorted(pts, key=lambda p: (p[0], -p[1]))
            frontier_ex = []
            max_y = -1.0
            for x, ex, cl in pts_sorted_ex:
                if ex > max_y:
                    frontier_ex.append((x, ex))
                    max_y = ex
            
            # Pareto frontier for cell acc
            pts_sorted_cl = sorted(pts, key=lambda p: (p[0], -p[2]))
            frontier_cl = []
            max_y = -1.0
            for x, ex, cl in pts_sorted_cl:
                if cl > max_y:
                    frontier_cl.append((x, cl))
                    max_y = cl
            
            # Plot all points as scatter
            ax1.scatter([p[0] for p in pts], [p[1] for p in pts], color=c, alpha=0.3, marker=m, s=30)
            ax2.scatter([p[0] for p in pts], [p[2] for p in pts], color=c, alpha=0.3, marker=m, s=30)
            
            # Plot Pareto frontier line (each tuple in frontier list contains only 2 elements: (x, y))
            if frontier_ex:
                ax1.plot([p[0] for p in frontier_ex], [p[1] for p in frontier_ex], color=c, label=f"{qname} (Frontier)", lw=2.5, marker=m)
            if frontier_cl:
                ax2.plot([p[0] for p in frontier_cl], [p[1] for p in frontier_cl], color=c, label=f"{qname} (Frontier)", lw=2.5, marker=m)
                
        for ax, title in [(ax1, "Exact Match Accuracy (Pass@1)"), (ax2, "Cell-Level Accuracy")]:
            ax.set_title(title, fontsize=12, fontweight="bold")
            ax.set_xlabel(x_label)
            ax.set_ylabel("Accuracy")
            ax.legend()
            ax.grid(True, alpha=0.3)
            
        plt.suptitle(f"Accuracy vs {title_suffix} (Sudoku Extreme)", fontsize=13, fontweight="bold", y=1.02)
        plt.tight_layout()
        plt.savefig(filename, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved: {filename}")

    # Plot 1: Compute (GFLOPs)
    plot_frontier("flops", "Compute Cost (GFLOPs)", "Compute Cost (GFLOPs)", "depth_quant_grid_flops.png")
    
    # Plot 2: Latency (ms/puzzle)
    plot_frontier("latency", "Inference Latency (ms/puzzle)", "Inference Latency", "depth_quant_grid_latency.png")


---
## Section 7 — Recursive State Similarity Analysis

**Novel diagnostic (no retraining needed):** We hook into the inner forward to collect
the `z_H` carry tensor at each H_cycle step, then measure cosine similarity between
consecutive recursive states. A well-trained model should show *decreasing* similarity
(each step refines the representation), while a saturated or collapsed model shows high similarity.

We compare FP32 vs INT4 to see if quantization forces carry states to collapse.


In [ ]:
# ── 7.1  Carry-state similarity hooks ────────────────────────────────────────
from models.recursive_reasoning.trm import (
    TinyRecursiveReasoningModel_ACTV1Carry,
    TinyRecursiveReasoningModel_ACTV1InnerCarry,
)

def collect_carry_similarities(mdl, loader, device="cpu", n_batches=5):
    """
    Hook inner.L_level to capture z_H tensors across all H_cycle & L_cycle passes.
    Returns mean cosine similarity between consecutive recursive layer outputs.

    A well-trained model: similarity decreases over steps (each step refines).
    A collapsed/over-quantized model: similarity stays high (no new information).
    """
    inner = get_inner(mdl)
    inner_model = inner.inner  # TinyRecursiveReasoningModel_ACTV1_Inner

    # Upcast bf16 init buffers for CPU eval
    if device == "cpu":
        for attr in ["H_init", "L_init"]:
            buf = getattr(inner_model, attr, None)
            if buf is not None and buf.dtype == torch.bfloat16:
                setattr(inner_model, attr, nn.Buffer(buf.float(), persistent=False))

    inner.eval()
    inner = inner.to(device)

    z_H_states = []

    def _hook(module, inp, out):
        z_H_states.append(out.detach().float().cpu())

    h = inner_model.L_level.register_forward_hook(_hook)

    sims_per_batch = []
    cast = lambda t: t.to(device).float() if (device == "cpu" and t.dtype == torch.bfloat16) else t.to(device)

    with torch.no_grad():
        for bi, (x_batch, y_true, pids) in enumerate(loader):
            if bi >= n_batches:
                break
            z_H_states.clear()

            x_batch = x_batch.to(device)
            y_true  = y_true.to(device)
            pids    = pids.to(device)
            batch = {
                "inputs":             x_batch.to(torch.int32),
                "labels":             y_true.to(torch.int32),
                "puzzle_identifiers": pids.to(torch.int32),
            }

            # Build and move carry properly
            carry = inner.initial_carry(batch)
            ic = carry.inner_carry
            carry = TinyRecursiveReasoningModel_ACTV1Carry(
                inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(
                    z_H=cast(ic.z_H), z_L=cast(ic.z_L),
                ),
                steps=carry.steps.to(device),
                halted=carry.halted.to(device),
                current_data={k: v.to(device) for k, v in carry.current_data.items()},
            )

            inner(carry, batch)  # one ACT forward pass; hooks fire on every L_level call

            if len(z_H_states) >= 2:
                sims = []
                for za, zb in zip(z_H_states[:-1], z_H_states[1:]):
                    a = za.mean(1)   # (B, D)
                    b = zb.mean(1)
                    cos = F.cosine_similarity(a, b, dim=-1).mean().item()
                    sims.append(cos)
                sims_per_batch.append(sims)

    h.remove()

    if not sims_per_batch:
        return []
    n_steps = min(len(s) for s in sims_per_batch)
    return [np.mean([s[i] for s in sims_per_batch]) for i in range(n_steps)]

print("Carry similarity collector defined.")


In [ ]:
# ── 7.2  Compare carry similarity across quantization variants ────────────────
sim_results = {}

if train_loader is not None and variants:
    # Compare a subset of variants (GPU ones — CPU INT8 is too slow for this)
    sim_targets = {k: v for k, v in variants.items()
                   if v[1] != "cpu"}  # skip CPU variants
    if not sim_targets:
        sim_targets = dict(list(variants.items())[:2])  # fallback: first two

    for vname, (vm, dev) in sim_targets.items():
        print(f"Collecting carry similarities for {vname} on {dev}...")
        try:
            sims = collect_carry_similarities(vm, train_loader, device=dev, n_batches=3)
            sim_results[vname] = sims
            if sims:
                print(f"  {vname}: {len(sims)} L_level calls, mean sim = {np.mean(sims):.4f}")
        except Exception as e:
            print(f"  [WARN] {vname}: {e}")
            sim_results[vname] = []
else:
    print("[SKIP] variants or train_loader not available.")

# Plot
if any(v for v in sim_results.values()):
    palette = ["#2196F3", "#00BCD4", "#4CAF50", "#FF9800", "#FF5722"]
    fig, ax = plt.subplots(figsize=(11, 5))
    for (vname, sims), color in zip(sim_results.items(), palette):
        if sims:
            ax.plot(range(1, len(sims)+1), sims, marker="o",
                    label=vname, color=color, lw=2)
    ax.axhline(0, ls="--", color="gray", alpha=0.5)
    ax.set_title("Cosine Similarity Between Consecutive Recursive Carry States",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("L_level Call Index (across H_cycles × L_cycles steps)")
    ax.set_ylabel("Mean Cosine Similarity (↓ = more specialised)")
    ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("carry_similarity_arc.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: carry_similarity_arc.png")


---
## Section 8 — Model Size & SRAM Footprint Estimator

This section maps the Sudoku TRM's compression landscape to realistic embedded hardware targets.

| Target device | SRAM budget | Example |
|---|---|---|
| Ultra-edge | < 256 KB | STM32F4, ESP32 |
| Standard TinyML | < 1 MB | ESP32-S3, Arduino Portenta H7 |
| Micro-edge | < 4 MB | Raspberry Pi Zero |


In [ ]:
# ── 8.1  Comprehensive model footprint analysis ───────────────────────────────
if 'model' in dir():
    inner = get_inner(model)

    # Split accounting: parameters vs buffers (puzzle emb lives in buffers)
    param_numel = sum(p.numel() for p in inner.parameters())
    buf_numel   = sum(b.numel() for b in inner.buffers())

    print(f"{'Component':<30} {'FP32 KB':>12} {'INT8 KB':>12} {'INT4 KB':>12}")
    print("-" * 70)

    # Transformer backbone (parameters)
    for label, numel, bits_list in [
        ("Transformer backbone (params)", param_numel, [32, 8, 4]),
    ]:
        vals = [numel * b / 8 / 1024 for b in bits_list]
        print(f"  {label:<28} {vals[0]:>12,.1f} {vals[1]:>12,.1f} {vals[2]:>12,.1f}")

    # Buffers — break down individually
    print()
    print(f"  {'Buffer name':<40} {'dtype':<10} {'KB':>10}")
    print("  " + "-" * 62)
    buf_total_kb = 0
    for name, buf in inner.named_buffers():
        kb = buf.numel() * buf.element_size() / 1024
        buf_total_kb += kb
        if kb > 1:
            print(f"  {name:<40} {str(buf.dtype):<10} {kb:>10,.1f}")
    print(f"  {'Total buffers':<40} {'':10} {buf_total_kb:>10,.1f}")

    print()
    # Inference-only footprint (no grad accumulators needed at deployment)
    # Puzzle emb is 2 KB  |  Grad accum (f32) — NOT needed at inference
    puzzle_emb_kb = 0
    for name, buf in inner.named_buffers():
        if 'puzzle_emb' in name or 'emb' in name.lower():
            puzzle_emb_kb += buf.numel() * buf.element_size() / 1024

    deploy_fp32_kb = param_numel * 32 / 8 / 1024 + puzzle_emb_kb
    deploy_int8_kb = param_numel *  8 / 8 / 1024 + puzzle_emb_kb
    deploy_int4_kb = param_numel *  4 / 8 / 1024 + puzzle_emb_kb

    print(f"{'Deployable size (no grad buffers)':<30} {'FP32 KB':>12} {'INT8 KB':>12} {'INT4 KB':>12}")
    print("-" * 70)
    for label, kb in [("Backbone + puzzle emb", None)]:
        print(f"  {'FP32':<28} {deploy_fp32_kb:>12,.1f}")
        print(f"  {'INT8':<28} {deploy_int8_kb:>12,.1f}")
        print(f"  {'INT4':<28} {deploy_int4_kb:>12,.1f}")

    print(f"\n  Target: < 1 MB (1024 KB) SRAM — backbone at INT4 = {param_numel*4/8/1024:.1f} KB")

    # DataFrame for plots
    import pandas as pd
    precisions = [("FP32", 32), ("INT8", 8), ("INT4", 4), ("INT2", 2)]
    rows = []
    for prec_name, bits in precisions:
        backbone_kb = param_numel * bits / 8 / 1024
        rows.append({
            "Precision":   prec_name,
            "Backbone_KB": backbone_kb,
            "Deploy_KB":   backbone_kb + puzzle_emb_kb,
            "fits_1MB":    "✓" if backbone_kb < 1024  else "✗",
            "fits_4MB":    "✓" if backbone_kb < 4096  else "✗",
        })
    df_footprint = pd.DataFrame(rows)
    print()
    print(df_footprint.to_string(index=False, float_format="{:.1f}".format))
else:
    print("[SKIP] model not loaded.")
    df_footprint = pd.DataFrame()


In [ ]:
df_footprint

In [ ]:
# ── 8.2  SRAM footprint visualisation ────────────────────────────────────────
if not df_footprint.empty:
    palette  = {"FP32": "#2196F3", "INT8": "#4CAF50", "INT4": "#FF9800", "INT2": "#F44336"}
    fig, ax  = plt.subplots(figsize=(10, 5))

    precs  = df_footprint["Precision"].tolist()
    sizes  = (df_footprint["Deploy_KB"]/1024).tolist()
    colors = [palette[p] for p in precs]

    bars = ax.bar(precs, sizes, color=colors, edgecolor="black", lw=0.8, alpha=0.85)
    for bar, sz in zip(bars, sizes):
        ax.text(bar.get_x() + bar.get_width()/2, sz + 0.2,
                f"{sz:.2f} MB", ha="center", va="bottom", fontsize=10)

    ax.axhline(0.25, ls=":",  color="red",    alpha=0.7, lw=1.5, label="256 KB (STM32F4)")
    ax.axhline(1.0,  ls="--", color="orange", alpha=0.7, lw=1.5, label="1 MB (ESP32-S3)")
    ax.axhline(4.0,  ls="-.", color="gray",   alpha=0.7, lw=1.5, label="4 MB (RPi Zero)")

    ax.set_title("Sudoku TRM SRAM Footprint by Precision", fontsize=13, fontweight="bold")
    ax.set_xlabel("Precision"); ax.set_ylabel("Model Size (MB)")
    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.3f}"))
    ax.legend(); ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.savefig("sram_footprint_arc.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: sram_footprint_arc.png")


In [ ]:
# ── 8.3  Accuracy vs SRAM: the key publishable plot ──────────────────────────
if results and not df_footprint.empty:
    # Map quant name to bit-width for size lookup
    bits_map = {"FP32": 32, "INT8": 8, "INT4 (fake)": 4}
    acc_vs_sram = []
    for vname, r in results.items():
        bits = bits_map.get(vname, 32)
        n    = sum(p.numel() for p in get_inner(model).parameters())
        mb   = n * bits / 8 / 1024 / 1024
        acc_vs_sram.append((vname, mb, r["exact_acc"]))

    fig, ax = plt.subplots(figsize=(10, 6))
    colors_p = ["#2196F3", "#4CAF50", "#FF5722"]
    for i, (label, mem_mb, acc) in enumerate(acc_vs_sram):
        ax.scatter(mem_mb, acc, s=200, zorder=5, color=colors_p[i % len(colors_p)])
        ax.annotate(f"{label}\n{acc:.3f}", (mem_mb, acc),
                    textcoords="offset points", xytext=(8, 4), fontsize=9)

    ax.axvline(1.0,  ls="--", color="orange", alpha=0.7, lw=1.5, label="1 MB SRAM")
    ax.axvline(0.25, ls=":",  color="red",    alpha=0.7, lw=1.5, label="256 KB SRAM")
    ax.set_title("Accuracy vs SRAM Footprint — Sudoku Extreme Edge Feasibility",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Model Memory (MB, log scale)")
    ax.set_ylabel("Exact Match Accuracy")
    ax.set_xscale("log"); ax.set_ylim(-0.05, 1.05)
    ax.legend(); ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.savefig("accuracy_vs_sram_arc.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: accuracy_vs_sram_arc.png")


---
## Summary & Next Steps

### Results Collected
| Experiment | Key Metric | Variable |
|---|---|---|
| Reasoning Decay (§5) | Δexact FP32→INT4 | `results` dict |
| Depth × Quant (§6) | Best H_cycles per precision | `grid_results` dict |
| Carry Similarity (§7) | Cosine sim across recursive steps | `sim_results` dict |
| SRAM Feasibility (§8) | Fits <1 MB? | `df_footprint` DataFrame |

### Recommended Next Steps

1. **INT8-aware checkpoint**: Re-export the checkpoint with explicit INT8 calibration using `torch.ao.quantization` for tighter accuracy recovery.
2. **Quantization-Aware Fine-tuning**: Run a short fine-tuning pass with `FakeQuantINT4` applied from the start — likely recovers most of the accuracy gap.
3. **On-device export**: Convert to ONNX / TFLite and measure real latency on an ESP32-S3 or Cortex-M7.
4. **Structured pruning**: Reduce `L_layers` or `hidden_size` and fine-tune — explore the accuracy-vs-SRAM frontier more granularly.
5. **ACT halting analysis**: Measure distribution of `carry.steps` at inference — do compressed models halt earlier (fewer recursive steps)?

*Based on: Jolicoeur-Martineau (2025), "Less is More: Recursive Reasoning with Tiny Networks", arXiv:2510.04871*


---
## Section 9 — Puzzle Embedding Compression

The puzzle embedding buffer (`puzzle_emb.weights`) is only 2 KB (1 row) at FP32 and has negligible deploy size.
The backbone at INT4 is only ~3.3 MB. This section compresses the embedding with three strategies:

| Strategy | Description | Expected size |
|---|---|---|
| **INT8 quantize** | Per-row symmetric INT8 of the weight buffer | ~15 MB |
| **SVD rank-r** | Low-rank factorization of weight matrix | r×(N+D)/N×D × FP32 |
| **Single-puzzle** | Load only 1 embedding row at inference time | ~0 MB overhead |


In [ ]:
# ── 9.1  Inspect puzzle_emb shape & byte count ───────────────────────────────
if 'model' in dir():
    inner = get_inner(model)
    pw = inner.puzzle_emb.weights          # (num_identifiers, emb_dim)  FP32
    N, D = pw.shape
    fp32_kb = pw.numel() * 4 / 1024
    int8_kb = pw.numel() * 1 / 1024
    print(f"puzzle_emb.weights  : {N} identifiers × {D} dims")
    print(f"  FP32 size         : {fp32_kb:,.1f} KB  ({fp32_kb/1024:.1f} MB)")
    print(f"  INT8 size (est.)  : {int8_kb:,.1f} KB  ({int8_kb/1024:.1f} MB)")

    lw = inner.puzzle_emb.local_weights    # (batch_size, emb_dim)
    print(f"puzzle_emb.local_weights : {lw.shape}  (non-persistent)")
else:
    print("[SKIP] model not loaded.")


In [ ]:
# ── 9.2  Option A — INT8 quantize the embedding buffer ───────────────────────
import copy

class INT8PuzzleEmbedding(nn.Module):
    """
    Drop-in replacement for CastedSparseEmbedding at eval time.
    Weights stored as INT8; scale per-row (symmetric).
    Forward dequantizes on the fly for selected rows only.
    """
    def __init__(self, weights_fp32: torch.Tensor, cast_to: torch.dtype):
        super().__init__()
        self.cast_to = cast_to
        # Per-row symmetric quantization
        absmax = weights_fp32.float().abs().amax(dim=1, keepdim=True).clamp(min=1e-8)
        scale  = absmax / 127.0                      # (N, 1) float32
        w_int8 = (weights_fp32.float() / scale).round().clamp(-127, 127).to(torch.int8)
        # Store as buffers (not parameters)
        self.register_buffer("w_int8", w_int8)       # (N, D) int8
        self.register_buffer("scale",  scale.squeeze(1))  # (N,) float32

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        rows_i8 = self.w_int8[inputs]                 # (B, D) int8
        sc      = self.scale[inputs].unsqueeze(-1)    # (B, 1) float32
        return (rows_i8.float() * sc).to(self.cast_to)

def apply_int8_puzzle_emb(mdl):
    """Replace puzzle_emb.weights buffer with INT8 version. Returns new model copy."""    
    m = copy.deepcopy(get_inner(mdl))
    pw = m.inner.puzzle_emb.weights
    cast_to = m.inner.puzzle_emb.cast_to
    new_emb = INT8PuzzleEmbedding(pw.cpu().float(), cast_to)
    if pw.is_cuda:
        new_emb = new_emb.cuda()
    m.inner.puzzle_emb = new_emb
    return m

if 'model' in dir():
    print("Applying INT8 puzzle embedding...")
    model_int8emb = apply_int8_puzzle_emb(model)

    # Verify output similarity
    inner    = get_inner(model)
    inner_q  = get_inner(model_int8emb)
    test_ids = torch.zeros(4, dtype=torch.int32).to(DEVICE)
    with torch.no_grad():
        orig = inner.inner.puzzle_emb(test_ids)
        qout = inner_q.inner.puzzle_emb(test_ids)
    cos = torch.nn.functional.cosine_similarity(orig.float(), qout.float(), dim=-1).mean()
    print(f"  Cosine similarity FP32 vs INT8 emb: {cos:.5f}")

    # Size comparison
    import io
    def buf_size_kb(m):
        buf = io.BytesIO()
        torch.save(get_inner(m).state_dict(), buf)
        return buf.tell() / 1024

    orig_kb = buf_size_kb(model)
    q_kb    = buf_size_kb(model_int8emb)
    print(f"  Original state_dict : {orig_kb:,.1f} KB  ({orig_kb/1024:.1f} MB)")
    print(f"  INT8-emb state_dict : {q_kb:,.1f} KB  ({q_kb/1024:.1f} MB)")
    print(f"  Reduction           : {(1 - q_kb/orig_kb)*100:.1f}%")
else:
    print("[SKIP] model not loaded.")


In [ ]:
# ── 9.3  Option B — SVD Low-Rank Puzzle Embedding ────────────────────────────
class SVDPuzzleEmbedding(nn.Module):
    """
    Stores puzzle embeddings as low-rank factors: W ≈ U @ V^T
    where U: (N, r), V: (D, r).
    At inference: row i = U[i] @ V^T  (r dot products per dim, not D).
    """
    def __init__(self, weights_fp32: torch.Tensor, rank: int, cast_to: torch.dtype):
        super().__init__()
        self.cast_to = cast_to
        W = weights_fp32.float()
        U, S, Vh = torch.linalg.svd(W, full_matrices=False)
        # Keep top-r components: W ≈ (U[:, :r] * S[:r]) @ Vh[:r, :]
        U_r  = (U[:, :rank] * S[:rank])   # (N, r)
        Vh_r = Vh[:rank, :]               # (r, D)
        self.register_buffer("U",  U_r)
        self.register_buffer("Vh", Vh_r)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        rows = self.U[inputs] @ self.Vh   # (B, D) fp32
        return rows.to(self.cast_to)

def apply_svd_puzzle_emb(mdl, rank: int):
    m = copy.deepcopy(get_inner(mdl))
    pw = m.inner.puzzle_emb.weights
    cast_to = m.inner.puzzle_emb.cast_to
    new_emb = SVDPuzzleEmbedding(pw.cpu().float(), rank=rank, cast_to=cast_to)
    if pw.is_cuda:
        new_emb = new_emb.cuda()
    m.inner.puzzle_emb = new_emb
    return m

if 'model' in dir():
    inner = get_inner(model)
    pw    = inner.inner.puzzle_emb.weights
    N, D  = pw.shape
    fp32_kb = pw.numel() * 4 / 1024

    print(f"SVD low-rank analysis  (W shape: {N}×{D}  FP32={fp32_kb:,.1f} KB)")
    print(f"{'Rank':>8} {'Size KB':>10} {'Reduction':>12} {'Cosine Sim':>12}")
    print("-" * 46)
    svd_models = {}
    test_ids = torch.zeros(4, dtype=torch.int32).to(DEVICE)
    with torch.no_grad():
        orig_out = inner.inner.puzzle_emb(test_ids).float()
    for rank in [8, 16, 32, 64, 128]:
        if rank >= min(N, D):
            continue
        m_svd = apply_svd_puzzle_emb(model, rank)
        svd_kb = (N * rank + rank * D) * 4 / 1024
        with torch.no_grad():
            svd_out = get_inner(m_svd).inner.puzzle_emb(test_ids).float()
        cos = torch.nn.functional.cosine_similarity(orig_out, svd_out, dim=-1).mean().item()
        print(f"{rank:>8} {svd_kb:>10,.1f} {(1-svd_kb/fp32_kb)*100:>11.1f}% {cos:>12.5f}")
        svd_models[rank] = m_svd
else:
    print("[SKIP] model not loaded.")
    svd_models = {}


In [ ]:
# ── 9.4  Option D — Single-Puzzle Inference (only load 1 row) ────────────────
class SinglePuzzleEmbedding(nn.Module):
    """
    For edge inference on a SINGLE known puzzle:
    pre-loads only that puzzle's embedding row.
    Keeps only the active puzzle's embedding row in SRAM.
    """
    def __init__(self, weights_fp32: torch.Tensor, puzzle_id: int, cast_to: torch.dtype):
        super().__init__()
        self.cast_to    = cast_to
        self.puzzle_id  = puzzle_id
        # Store only the one row needed
        row = weights_fp32[puzzle_id].clone()  # (D,)
        self.register_buffer("row", row)
        self.orig_weights = weights_fp32  # Keep track of original weights for downstream benchmarks

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        # inputs is ignored — we always return the pre-loaded row
        B = inputs.shape[0]
        return self.row.unsqueeze(0).expand(B, -1).to(self.cast_to)

def make_single_puzzle_model(mdl, puzzle_id: int):
    m = copy.deepcopy(get_inner(mdl))
    pw      = m.inner.puzzle_emb.weights
    cast_to = m.inner.puzzle_emb.cast_to
    new_emb = SinglePuzzleEmbedding(pw.cpu().float(), puzzle_id=puzzle_id, cast_to=cast_to)
    if pw.is_cuda:
        new_emb = new_emb.cuda()
    m.inner.puzzle_emb = new_emb
    return m

if 'model' in dir():
    m_single = make_single_puzzle_model(model, puzzle_id=0)
    import io
    buf = io.BytesIO(); torch.save(get_inner(m_single).state_dict(), buf)
    single_kb = buf.tell() / 1024
    inner = get_inner(model)
    N, D  = inner.inner.puzzle_emb.weights.shape
    backbone_kb = sum(p.numel() for p in inner.parameters()) * 4 / 1024
    print(f"Single-puzzle model state_dict : {single_kb:,.1f} KB  ({single_kb/1024:.2f} MB)")
    print(f"  Backbone params              : {backbone_kb:,.1f} KB")
    print(f"  Per-puzzle embedding row     : {D*4/1024:,.1f} KB")
    print(f"  Total (1 puzzle)             : {backbone_kb + D*4/1024:,.1f} KB")
    print(f"\nThis is the architecture-level solution: load puzzle embedding from flash,")
    print(f"keep only the active row in SRAM during inference.")
else:
    print("[SKIP] model not loaded.")


In [ ]:
# ── 9.5  Footprint summary across embedding strategies ───────────────────────
if 'model' in dir():
    import io, pandas as pd

    def sd_kb(m):
        buf = io.BytesIO()
        torch.save(get_inner(m).state_dict(), buf)
        return buf.tell() / 1024

    inner    = get_inner(model)
    N, D     = inner.inner.puzzle_emb.weights.shape
    back_kb  = sum(p.numel() for p in inner.parameters()) * 4 / 1024

    rows = [
        {"Strategy": "FP32 (baseline)",     "Emb KB": N*D*4/1024, "Backbone KB": back_kb,
         "Total KB": sd_kb(model),            "Fits 4MB": "✗"},
        {"Strategy": "INT8 emb",            "Emb KB": N*D*1/1024, "Backbone KB": back_kb,
         "Total KB": sd_kb(model_int8emb) if 'model_int8emb' in dir() else 0, "Fits 4MB": "✗"},
    ]
    for rank, m_svd in (svd_models.items() if svd_models else {}.items()):
        svd_kb = (N*rank + rank*D)*4/1024
        rows.append({"Strategy": f"SVD r={rank}", "Emb KB": svd_kb,
                     "Backbone KB": back_kb, "Total KB": back_kb + svd_kb,
                     "Fits 4MB": "✓" if (back_kb + svd_kb) < 4096 else "✗"})
    rows.append({"Strategy": "Single-puzzle", "Emb KB": D*4/1024,
                 "Backbone KB": back_kb, "Total KB": back_kb + D*4/1024,
                 "Fits 4MB": "✓" if (back_kb + D*4/1024) < 4096 else "✗"})
    rows.append({"Strategy": "INT4 backbone + single-puzzle",
                 "Emb KB": D*4/1024, "Backbone KB": back_kb/8,
                 "Total KB": back_kb/8 + D*4/1024,
                 "Fits 4MB": "✓" if (back_kb/8 + D*4/1024) < 4096 else "✗"})

    df = pd.DataFrame(rows)
    df["Total MB"] = df["Total KB"] / 1024
    print(df[["Strategy","Emb KB","Backbone KB","Total KB","Total MB","Fits 4MB"]].to_string(index=False, float_format="{:.1f}".format))
else:
    print("[SKIP] model not loaded.")


---
## Section 10 — Fixed Evaluation: Per-Puzzle Aggregation

The submission evaluator scores per-**puzzle** (best-of-2 attempts after majority vote across augmentations).
`evaluate_arc()` in §5 scored per-**sample** (each augmentation independently), giving ~1% exact match
vs the 68.89% submission score.

This section introduces `evaluate_arc_per_puzzle()` which implements the paper's true test-time search logic:
it crops the padding/EOS tokens, applies inverse data-augmentations, hashes the canonical grids,
and performs quality-weighted majority voting across all augmentations — matching the official submission evaluator's logic.

### Note: `evaluate_arc_per_puzzle` has been moved to Section 3 for global accessibility.

In [ ]:
# ── 10.2  High-performance direct on-the-fly per-puzzle TTA evaluator ─────────
@torch.no_grad()
def evaluate_arc_directly_from_source(mdl, test_puzzles_path, identifiers_path, device="cpu", n_sup_max=10, num_aug=128, batch_size=512):
    """
    On-the-fly, high-performance per-puzzle TTA evaluator.
    Loads the clean unaugmented JSON puzzles, performs augmentation on-the-fly in memory,
    batches predictions, and aggregates results.
    
    This completely eliminates:
    1. Disk I/O bottlenecks of loading massive flat dataset files.
    2. Missing/clamped identifier bugs (PIDs are resolved dynamically).
    3. Host-to-device dataset loading overhead.
    """
    import os
    import json
    import time
    import torch
    import numpy as np
    from tqdm import tqdm
    
    from models.recursive_reasoning.trm import (
        TinyRecursiveReasoningModel_ACTV1Carry,
        TinyRecursiveReasoningModel_ACTV1InnerCarry,
    )
    from dataset.build_arc_dataset import (
        aug, 
        inverse_aug, 
        grid_hash, 
        arc_grid_to_np, 
        np_grid_to_seq_translational_augment,
        PuzzleIdSeparator,
        SudokuAugmentRetriesFactor
    )
    from evaluators.arc import _crop
    
    inner = get_inner(mdl)
    inner.eval()
    inner = inner.to(device)

    # Load test puzzles and puzzle identifier map
    with open(test_puzzles_path, "r") as f:
        test_puzzles = json.load(f)
    with open(identifiers_path, "r") as f:
        identifier_map = json.load(f)
        
    # Build reverse map of identifiers
    identifier_to_pid = {name: pid for pid, name in enumerate(identifier_map)}

    local_preds = {}
    local_hmap = {}
    
    t0 = time.time()
    n_puzzles = len(test_puzzles)
    
    for name, puzzle in tqdm(test_puzzles.items(), desc="Evaluating puzzles directly"):
        # Precompute canonical test input/label hashes
        puzzle_test_hashes = []
        for pair in puzzle["test"]:
            inp_grid = arc_grid_to_np(pair["input"])
            out_grid = arc_grid_to_np(pair["output"])
            puzzle_test_hashes.append((grid_hash(inp_grid), grid_hash(out_grid)))
            
        # Get canonical examples
        examples = [(arc_grid_to_np(pair["input"]), arc_grid_to_np(pair["output"])) for pair in puzzle["test"]]
        
        # 1. Generate unique augmentations on-the-fly in-memory
        group = []
        hashes = set()
        
        # Helper function to compute augmented puzzle hash
        def get_aug_puzzle_hash(aug_examples):
            h_list = []
            for inp, out in aug_examples:
                h_list.append(f"{grid_hash(inp)}|{grid_hash(out)}")
            h_list.sort()
            import hashlib
            return hashlib.sha256("|".join(h_list).encode()).hexdigest()

        # Add canonical (unaugmented) puzzle first
        group.append((name, examples))
        hashes.add(get_aug_puzzle_hash(examples))
        
        # Generate TTA augmentations
        for _trial in range(SudokuAugmentRetriesFactor * num_aug):
            aug_name, map_grid = aug(name)
            aug_examples = [(map_grid(inp), map_grid(out)) for inp, out in examples]
            h = get_aug_puzzle_hash(aug_examples)
            if h not in hashes:
                hashes.add(h)
                group.append((aug_name, aug_examples))
            if len(group) >= num_aug + 1:
                break
                
        # 2. Build flat input batches for this puzzle's augmentations
        puzzle_inputs = []
        puzzle_labels = []
        puzzle_pids = []
        
        # Each item in group is: (aug_name, list of (inp, out) examples)
        # We store metadata to map each batched sample back to the original test example index and inverse function
        sample_meta = []
        
        for aug_name, aug_examples in group:
            # Map aug_name back to its integer PID embedding index
            pid = identifier_to_pid.get(aug_name, 0)
            if pid == 0:
                # If name is not in pre-trained vocabs, fallback to canonical
                pid = identifier_to_pid.get(name, 0)
                
            for idx_ex, (inp, out) in enumerate(aug_examples):
                inp_seq, out_seq = np_grid_to_seq_translational_augment(inp, out, do_translation=False)
                puzzle_inputs.append(inp_seq)
                puzzle_labels.append(out_seq)
                puzzle_pids.append(pid)
                sample_meta.append((aug_name, idx_ex))
                
        # Convert arrays to numpy stack
        puzzle_inputs = np.stack(puzzle_inputs, 0)
        puzzle_labels = np.stack(puzzle_labels, 0)
        puzzle_pids = np.array(puzzle_pids, dtype=np.int32)
        
        # 3. Process batches through the model
        num_samples = len(puzzle_inputs)
        local_preds[name] = {}
        
        for start_idx in range(0, num_samples, batch_size):
            end_idx = min(start_idx + batch_size, num_samples)
            B = end_idx - start_idx
            
            x_batch = torch.tensor(puzzle_inputs[start_idx:end_idx], dtype=torch.long, device=device)
            y_true  = torch.tensor(puzzle_labels[start_idx:end_idx], dtype=torch.long, device=device)
            pids    = torch.tensor(puzzle_pids[start_idx:end_idx], dtype=torch.long, device=device)
            
            batch = {
                "inputs":             x_batch.to(torch.int32),
                "labels":             y_true.to(torch.int32),
                "puzzle_identifiers": pids.to(torch.int32),
            }
            
            carry = inner.initial_carry(batch)
            ic    = carry.inner_carry
            cast  = lambda t: t.to(device)
            carry = TinyRecursiveReasoningModel_ACTV1Carry(
                inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(
                    z_H=cast(ic.z_H), z_L=cast(ic.z_L)),
                steps=carry.steps.to(device),
                halted=carry.halted.to(device),
                current_data={k: v.to(device) for k, v in carry.current_data.items()},
            )
            
            last_outputs = None
            for _ in range(n_sup_max):
                carry, outputs = inner(carry, batch)
                last_outputs = outputs
                if carry.halted.all():
                    break
                    
            if last_outputs is None:
                continue
                
            preds_batch = last_outputs["logits"].argmax(-1).cpu().numpy()  # (B, seq_len)
            q_logits    = last_outputs.get("q_halt_logits", torch.zeros(B, device=device))
            q_values    = q_logits.sigmoid().cpu().numpy().flatten()    # (B,)
            
            # Post-process this batch's predictions
            for b_i in range(B):
                global_idx = start_idx + b_i
                aug_name, idx_ex = sample_meta[global_idx]
                
                # Get the canonical input/label hashes for this test example
                inp_hash, lbl_hash = puzzle_test_hashes[idx_ex]
                
                # Resolve inverse transform for this specific augmentation
                _orig_name_decoded, _inverse_fn = inverse_aug(aug_name)
                
                pred_seq = preds_batch[b_i]
                q_val = float(q_values[b_i])
                
                # Crop and inverse transform prediction
                pred_grid = _inverse_fn(_crop(pred_seq))
                pred_hash = grid_hash(pred_grid)
                
                local_hmap[pred_hash] = pred_grid
                
                local_preds[name].setdefault(inp_hash, [])
                local_preds[name][inp_hash].append((pred_hash, q_val))
                
    elapsed = time.time() - t0
    
    # 4. Aggregated voting and accuracy evaluation
    pass_Ks = [1, 2, 5]
    correct = [0.0 for _ in pass_Ks]
    
    for name, puzzle in test_puzzles.items():
        num_test_correct = [0 for _ in pass_Ks]
        
        for idx_ex, pair in enumerate(puzzle["test"]):
            inp_grid = arc_grid_to_np(pair["input"])
            out_grid = arc_grid_to_np(pair["output"])
            
            input_hash = grid_hash(inp_grid)
            label_hash = grid_hash(out_grid)
            
            p_map = {}
            for h, q in local_preds.get(name, {}).get(input_hash, []):
                p_map.setdefault(h, [0, 0.0])
                p_map[h][0] += 1
                p_map[h][1] += q
                
            if not len(p_map):
                continue
                
            for h, stats in p_map.items():
                stats[1] /= stats[0]
                
            p_map_sorted = sorted(p_map.items(), key=lambda kv: kv[1], reverse=True)
            
            for i, k in enumerate(pass_Ks):
                ok = False
                for h, stats in p_map_sorted[:k]:
                    ok |= (h == label_hash)
                num_test_correct[i] += int(ok)
                
        for i in range(len(pass_Ks)):
            correct[i] += num_test_correct[i] / len(puzzle["test"])
            
    # Cell accuracy estimation
    cell_hits = 0
    n_cells = 0
    for name, puzzle in test_puzzles.items():
        for idx_ex, pair in enumerate(puzzle["test"]):
            inp_grid = arc_grid_to_np(pair["input"])
            out_grid = arc_grid_to_np(pair["output"])
            input_hash = grid_hash(inp_grid)
            
            preds_list = local_preds.get(name, {}).get(input_hash, [])
            if not preds_list:
                continue
                
            p_map = {}
            for h, q in preds_list:
                p_map.setdefault(h, [0, 0.0])
                p_map[h][0] += 1
                p_map[h][1] += q
            for h, stats in p_map.items():
                stats[1] /= stats[0]
                
            p_map_sorted = sorted(p_map.items(), key=lambda kv: kv[1], reverse=True)
            top_hash = p_map_sorted[0][0]
            top_grid = local_hmap[top_hash]
            
            if top_grid.shape == out_grid.shape:
                cell_hits += (top_grid == out_grid).sum()
                n_cells += out_grid.size
            else:
                n_cells += out_grid.size
                
    cell_acc = cell_hits / n_cells if n_cells > 0 else 0.0
    pass_1_acc = correct[0] / n_puzzles if n_puzzles > 0 else 0.0
    pass_2_acc = correct[1] / n_puzzles if n_puzzles > 0 else 0.0
    ms_per_puzzle = (elapsed / n_puzzles) * 1000 if n_puzzles > 0 else 0.0
    
    print(f"\nDirect Evaluation Results (All {n_puzzles} puzzles):")
    print(f"  Pass@1 Exact: {pass_1_acc:.4f}")
    print(f"  Pass@2 Exact: {pass_2_acc:.4f}")
    print(f"  Cell Acc    : {cell_acc:.4f}")
    print(f"  Latency     : {ms_per_puzzle:.2f} ms/puzzle")
    
    return pass_1_acc, pass_2_acc, cell_acc, ms_per_puzzle


In [ ]:
# # ── 10.3  Run the High-Performance Direct Source Evaluator ─────────────────────
# # FP32 Direct TTA Inference on CPU/GPU directly from original JSON files
# if 'model' in dir():
#     test_json = os.path.join(DATA_DIR, "test_puzzles.json")
#     ids_json = os.path.join(DATA_DIR, "identifiers.json")
    
#     if os.path.exists(test_json) and os.path.exists(ids_json):
#         # Evaluate FP32 model with 128 TTA augmentations, batch size 512
#         p1, p2, cell, lat = evaluate_arc_directly_from_source(
#             model,
#             test_json,
#             ids_json,
#             device="cuda" if torch.cuda.is_available() else "cpu",
#             n_sup_max=10,
#             num_aug=128,
#             batch_size=512
#         )
#     else:
#         print("test_puzzles.json or identifiers.json missing in DATA_DIR.")


In [ ]:
# ── 10.2  Re-evaluate all variants with per-puzzle aggregation ────────────────
if 'variants' in dir() and variants and test_loader is not None:
    print("Per-puzzle evaluation (All batches, ~1280 samples)...\n")
    print(f"{'Variant':<18} {'Pass@1 Exact':>14} {'Pass@2 Exact':>14} {'Cell Acc':>10} {'ms/puzzle':>12} {'N puzzles':>10}")
    print("-" * 82)

    pp_results = {}
    for vname, (vm, dev) in variants.items():
        try:
            p1, p2, cacc, ms, npuzz = evaluate_arc_per_puzzle(
                vm, test_loader, device=dev, n_sup_max=10, max_batches=None, return_pass2=True)
            pp_results[vname] = {"pass_1_exact": p1, "pass_2_exact": p2, "cell_acc": cacc,
                                 "latency_ms": ms, "n_puzzles": npuzz}
            print(f"  {vname:<16} {p1:>14.4f} {p2:>14.4f} {cacc:>10.4f} {ms:>12.2f} {npuzz:>10}")
        except Exception as e:
            print(f"  {vname:<16} ERROR: {e}")

    # Also evaluate INT8-emb model if available
    if 'model_int8emb' in dir():
        try:
            p1, p2, cacc, ms, npuzz = evaluate_arc_per_puzzle(
                model_int8emb, test_loader, device=str(DEVICE), n_sup_max=10, max_batches=None, return_pass2=True)
            pp_results["INT8-emb"] = {"pass_1_exact": p1, "pass_2_exact": p2, "cell_acc": cacc,
                                      "latency_ms": ms, "n_puzzles": npuzz}
            print(f"  {'INT8-emb':<16} {p1:>14.4f} {p2:>14.4f} {cacc:>10.4f} {ms:>12.2f} {npuzz:>10}")
        except Exception as e:
            print(f"  INT8-emb ERROR: {e}")

else:
    print("[SKIP] variants or test_loader not available.")


---
## Section 11 — Calibrated INT4 Quantization

The fake INT4 in §4 used symmetric per-tensor quantization with no calibration.
This destroyed the model (0% exact match, carry similarity 0.72).

This section implements **per-channel asymmetric INT4** with calibration statistics
collected from training data — matching production-quality quantization pipelines.


In [ ]:
# ── 11.1  Per-channel calibration statistics ──────────────────────────────────
class ChannelCalibrator:
    """Collect per-output-channel (per-row) min/max statistics over calibration batches."""
    def __init__(self):
        self.min_vals = None
        self.max_vals = None
        self.n_batches = 0

    def update(self, weight: torch.Tensor):
        """weight: (out, in) — collect stats over out dimension."""
        w = weight.float()
        row_min = w.min(dim=1).values   # (out,)
        row_max = w.max(dim=1).values   # (out,)
        if self.min_vals is None:
            self.min_vals = row_min
            self.max_vals = row_max
        else:
            self.min_vals = torch.minimum(self.min_vals, row_min)
            self.max_vals = torch.maximum(self.max_vals, row_max)
        self.n_batches += 1

    def get_scale_zp(self, n_bits=4):
        """Return per-channel (scale, zero_point) for asymmetric quantization."""
        q_min = -(2 ** (n_bits - 1))
        q_max =  (2 ** (n_bits - 1)) - 1
        scale = (self.max_vals - self.min_vals).clamp(min=1e-8) / (q_max - q_min)
        zp    = (q_min - self.min_vals / scale).round().clamp(q_min, q_max)
        return scale, zp.to(torch.int8)


class CalibratedFakeQuantINT4(nn.Module):
    """Asymmetric per-channel INT4 fake-quant wrapper.
    Uses calibrated scale/zero_point instead of symmetric per-tensor.
    """
    def __init__(self, weight: torch.Tensor, bias=None, n_bits=4):
        super().__init__()
        cal = ChannelCalibrator()
        cal.update(weight)
        scale, zp = cal.get_scale_zp(n_bits)

        self.n_bits  = n_bits
        self.q_min   = -(2 ** (n_bits - 1))
        self.q_max   =  (2 ** (n_bits - 1)) - 1
        self.register_buffer("scale", scale)   # (out,)
        self.register_buffer("zp",    zp)      # (out,) int8
        # Fake-quantize weight at construction time (stateless forward)
        w_fq = self._fake_quant(weight.float())
        self.weight = nn.Parameter(w_fq.to(weight.dtype), requires_grad=False)
        self.bias   = nn.Parameter(bias.clone(), requires_grad=False) if bias is not None else None

    def _fake_quant(self, w: torch.Tensor) -> torch.Tensor:
        """Per-row asymmetric quantization then dequantization."""
        sc = self.scale.unsqueeze(1)   # (out, 1)
        zp = self.zp.float().unsqueeze(1)
        w_q = ((w / sc) + zp).round().clamp(self.q_min, self.q_max)
        return (w_q - zp) * sc        # dequantized

    def forward(self, x):
        w = self.weight.to(x.dtype)
        b = self.bias.to(x.dtype) if self.bias is not None else None
        return F.linear(x, w, b)


def quantize_int4_calibrated(mdl: nn.Module) -> nn.Module:
    """Replace Linear/CastedLinear with calibrated asymmetric INT4 fake-quant."""
    from models.layers import CastedLinear
    m = copy.deepcopy(get_inner(mdl))
    replaced = 0
    for name, module in list(m.named_modules()):
        if not isinstance(module, (nn.Linear, CastedLinear)):
            continue
        parts  = name.split(".")
        parent = m
        for p in parts[:-1]:
            parent = getattr(parent, p)
        setattr(parent, parts[-1],
                CalibratedFakeQuantINT4(module.weight, module.bias))
        replaced += 1
    print(f"  Replaced {replaced} layers → CalibratedFakeQuantINT4 (per-channel, asymmetric)")
    return m

print("Calibrated INT4 helpers defined.")


In [ ]:
# # ── 11.2  Apply calibrated INT4 and evaluate ──────────────────────────────────
# if 'model' in dir() and train_loader is not None:
#     print("Creating calibrated INT4 model...")
#     model_cal_int4 = quantize_int4_calibrated(model)
#     if torch.cuda.is_available():
#         model_cal_int4 = model_cal_int4.cuda()

#     # Carry similarity check (compare to naive fake-quant)
#     from collections import defaultdict

#     def quick_carry_sim(mdl, loader, device, n_batches=3):
#         """Single-batch carry similarity metric."""
#         from models.recursive_reasoning.trm import (
#             TinyRecursiveReasoningModel_ACTV1Carry,
#             TinyRecursiveReasoningModel_ACTV1InnerCarry,
#         )
#         inner = get_inner(mdl)
#         inner.eval()
#         inner = inner.to(device)
#         sims = []
#         for i, (x, y, pids) in enumerate(loader):
#             if i >= n_batches: break
#             x, y, pids = x.to(device), y.to(device), pids.to(device)
#             batch = {"inputs": x.to(torch.int32), "labels": y.to(torch.int32),
#                      "puzzle_identifiers": pids.to(torch.int32)}
#             carry = inner.initial_carry(batch)
#             ic = carry.inner_carry
#             carry = TinyRecursiveReasoningModel_ACTV1Carry(
#                 inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(
#                     z_H=ic.z_H.to(device), z_L=ic.z_L.to(device)),
#                 steps=carry.steps.to(device), halted=carry.halted.to(device),
#                 current_data={k: v.to(device) for k, v in carry.current_data.items()})
#             prev_z = None
#             for _ in range(6):
#                 carry, _ = inner(carry, batch)
#                 z = carry.inner_carry.z_H
#                 if prev_z is not None:
#                     sim = F.cosine_similarity(z.float().flatten(1), prev_z.float().flatten(1), dim=1).mean().item()
#                     sims.append(sim)
#                 prev_z = z.detach().clone()
#                 if carry.halted.all(): break
#         return float(np.mean(sims)) if sims else 0.0

#     dev = str(DEVICE)
#     sim_fp32 = quick_carry_sim(model,         train_loader, dev)
#     sim_naive = quick_carry_sim(
#         variants["INT4 (fake)"][0] if 'variants' in dir() and "INT4 (fake)" in variants else model,
#         train_loader, dev)
#     sim_cal  = quick_carry_sim(model_cal_int4, train_loader, dev)

#     print(f"\nCarry similarity comparison:")
#     print(f"  FP32 (bf16)            : {sim_fp32:.4f}  (baseline — lower = more refinement)")
#     print(f"  INT4 naive (§4)        : {sim_naive:.4f}  (higher = carry collapsed)")
#     print(f"  INT4 calibrated (§11)  : {sim_cal:.4f}")

#     # Per-puzzle accuracy
#     print("\nPer-puzzle evaluation (calibrated INT4, 30 batches)...")
#     pexact, cacc, ms, npuzz = evaluate_arc_per_puzzle(
#         model_cal_int4, train_loader, device=dev, n_sup_max=10, max_batches=None)
#     print(f"  Puzzle exact : {pexact:.4f}")
#     print(f"  Cell acc     : {cacc:.4f}")
#     print(f"  ms/puzzle    : {ms:.2f}")
# else:
#     print("[SKIP] model or train_loader not available.")


---
## Section 12 — Quantization-Aware Fine-tuning (QAT)

A short fine-tuning pass with fake-quant active in the forward path forces the model
to learn representations that are robust to INT4 noise. Even 50–100 gradient steps
can recover most of the accuracy gap.


In [ ]:
# # ── 12.1  QAT fine-tuning loop ────────────────────────────────────────────────
# def run_qat(base_mdl, loader, n_steps=50, lr=1e-5, device="cuda"):
#     """
#     Short QAT loop: apply INT4 fake-quant to all linear weights,
#     then fine-tune for n_steps to recover quantization-induced accuracy loss.
#     Uses the ACT loss from the model's own forward pass.
#     """
#     from models.recursive_reasoning.trm import (
#         TinyRecursiveReasoningModel_ACTV1Carry,
#         TinyRecursiveReasoningModel_ACTV1InnerCarry,
#     )

#     # Start from calibrated INT4 model so we're already near the minima
#     m = copy.deepcopy(get_inner(base_mdl)).to(device)

#     # Make all fake-quant weights trainable
#     for name, module in m.named_modules():
#         if isinstance(module, CalibratedFakeQuantINT4):
#             module.weight.requires_grad_(True)

#     optimizer = torch.optim.AdamW(
#         [p for p in m.parameters() if p.requires_grad], lr=lr, weight_decay=0.01)

#     m.train()
#     data_iter = iter(loader)
#     losses = []

#     print(f"QAT fine-tuning for {n_steps} steps (lr={lr}) with micro-batching (prevent OOM)...")
#     for step in range(n_steps):
#         try:
#             x, y, pids = next(data_iter)
#         except StopIteration:
#             data_iter = iter(loader)
#             x, y, pids = next(data_iter)

#         # Micro-batching to fit in 4GB VRAM
#         micro_batch_size = 16
#         num_micro_batches = max(1, len(x) // micro_batch_size)
        
#         optimizer.zero_grad()
#         accum_loss = 0.0
        
#         for mb_idx in range(num_micro_batches):
#             mb_start = mb_idx * micro_batch_size
#             mb_end = mb_start + micro_batch_size
            
#             mb_x = x[mb_start:mb_end].to(device)
#             mb_y = y[mb_start:mb_end].to(device)
#             mb_pids = pids[mb_start:mb_end].to(device)
            
#             batch = {"inputs": mb_x.to(torch.int32), "labels": mb_y.to(torch.int32),
#                      "puzzle_identifiers": mb_pids.to(torch.int32)}

#             carry = m.initial_carry(batch)
#             ic = carry.inner_carry
#             carry = TinyRecursiveReasoningModel_ACTV1Carry(
#                 inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(
#                     z_H=ic.z_H.to(device), z_L=ic.z_L.to(device)),
#                 steps=carry.steps.to(device), halted=carry.halted.to(device),
#                 current_data={k: v.to(device) for k, v in carry.current_data.items()})

#             mb_loss = torch.tensor(0.0, device=device)
#             for sup_step in range(4):   # 4 supervision steps (fast)
#                 carry, outputs = m(carry, batch)
#                 logits = outputs["logits"]   # (B, seq, vocab)
#                 labels = mb_y.long()
#                 loss   = F.cross_entropy(
#                     logits.reshape(-1, logits.size(-1)),
#                     labels.reshape(-1), ignore_index=-100)
#                 mb_loss = mb_loss + loss
#                 if carry.halted.all(): break

#             mb_loss_normalized = mb_loss / num_micro_batches
#             mb_loss_normalized.backward()
#             accum_loss += mb_loss.item()

#         torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
#         optimizer.step()
#         losses.append(accum_loss)

#         if (step + 1) % 10 == 0:
#             print(f"  Step {step+1:3d}/{n_steps} | loss={np.mean(losses[-10:]):.4f}")

#     m.eval()
#     return m

# # Clean up old training variables to free up GPU memory
# import gc
# for var in ['qat_model', 'qat_model_v', 'fused_model', 'cpu_model', 'wrapper_cpu']:
#     if var in globals():
#         try:
#             del globals()[var]
#         except Exception:
#             pass
# gc.collect()
# torch.cuda.empty_cache()

# # if 'model' in dir() and train_loader is not None:
# print('QAT fine-tuning commented out.')
# # [COMMENTED OUT FOR MODAL EXECUTION TO PREVENT SLOW EXPERIMENTS]
# #     qat_model = run_qat(model, train_loader, n_steps=50, lr=1e-5, device=str(DEVICE))
# #     print("
# # QAT complete. Evaluating...")
# #     pexact, cacc, ms, npuzz = evaluate_arc_per_puzzle(
# #         qat_model, train_loader, device=str(DEVICE), n_sup_max=10, max_batches=None)
# #     print(f"  QAT INT4 puzzle exact : {pexact:.4f}")
# #     print(f"  QAT INT4 cell acc     : {cacc:.4f}")
# # else:
# #     print("[SKIP] model or train_loader not available.")


---
## Section 13 — Structured Pruning

Structural pruning removes entire rows/columns from weight matrices,
reducing both parameter count and activation memory. We evaluate
magnitude-based row pruning at 25% and 50% sparsity levels.


In [ ]:
# ── 13.1  Magnitude-based row pruning ────────────────────────────────────────
def prune_linear_rows(mdl, prune_ratio=0.25):
    """
    Prune the lowest-L2-norm rows of each Linear/CastedLinear layer.
    This is unstructured weight masking (sets rows to zero) — preserves
    architecture shape but reduces effective computation on sparse hardware.
    Returns model with zero-masked rows and sparsity stats.
    """
    from models.layers import CastedLinear
    m = copy.deepcopy(get_inner(mdl))
    total_weights = 0
    total_pruned  = 0

    for name, module in m.named_modules():
        if not isinstance(module, (nn.Linear, CastedLinear)):
            continue
        
        w = module.weight.data   # (out, in)
        total_weights += w.numel()
        
        # Skip output heads (vocab mapping and halting prediction) to avoid catastrophic failure
        if "lm_head" in name or "q_head" in name:
            continue

        with torch.no_grad():
            row_norms = w.norm(dim=1)   # (out,)
            threshold = torch.quantile(row_norms, prune_ratio)
            mask = (row_norms >= threshold).unsqueeze(1).float()  # (out, 1)
            module.weight.data = w * mask
            if module.bias is not None:
                b_mask = (row_norms >= threshold).float()
                module.bias.data = module.bias.data * b_mask

        n_pruned = (row_norms < threshold).sum().item()
        total_pruned += int(n_pruned * w.shape[1])

    sparsity = total_pruned / total_weights if total_weights > 0 else 0.0
    return m, sparsity

if 'model' in dir() and test_loader is not None:
    # Recover puzzle_emb.forward to original state if it was corrupted/patched in memory by interrupted runs
    inner_model = get_inner(model)
    if hasattr(inner_model, "puzzle_emb"):
        from models.sparse_embedding import CastedSparseEmbedding
        inner_model.puzzle_emb.forward = CastedSparseEmbedding.forward.__get__(inner_model.puzzle_emb, CastedSparseEmbedding)

    print(f"{'Prune ratio':>14} {'Actual sparsity':>18} {'Puzzle Exact':>14} {'Cell Acc':>10}")
    print("-" * 60)
    pruned_results = {}
    for ratio in [0.0, 0.25, 0.50]:
        m_pruned, sparsity = prune_linear_rows(model, prune_ratio=ratio)
        if torch.cuda.is_available():
            m_pruned = m_pruned.cuda()
        pexact, cacc, ms, npuzz = evaluate_arc_per_puzzle(
            m_pruned, test_loader, device=str(DEVICE), n_sup_max=10, max_batches=None, fast_mode=True)
        pruned_results[ratio] = {"sparsity": sparsity, "puzzle_exact": pexact, "cell_acc": cacc}
        print(f"  {ratio:>12.0%} {sparsity:>18.2%} {pexact:>14.4f} {cacc:>10.4f}")
else:
    print("[SKIP] model or test_loader not available.")
    pruned_results = {}


---
## Section 14 — TorchScript Export (Edge-Native Serialization)

`torch.onnx.export` hangs on recursive models: it traces the full unrolled graph for a
81-token sequence, generating millions of nodes — taking hours or never finishing.

**TorchScript** (`torch.jit.trace`) is the correct tool for PyTorch-native edge deployment:
- Completes in **seconds**, not hours
- Produces `.pt` files readable by **PyTorch Mobile**, **ExecuTorch**, and **LibTorch (C++)**
- Supports dynamic batch size natively
- Can be converted to CoreML (Apple) or TFLite via `ai_edge_torch`

We export the backbone step (puzzle_emb external, same design as §14.1).


In [ ]:
%uv pip install onnx onnxruntime

In [ ]:
# ── 14.1  TorchScript trace of single TRM backbone step ──────────────────────
#
# torch.onnx.export hangs because it unrolls the full recursive graph for
# a 81-token sequence → millions of ONNX nodes. Use torch.jit.trace instead.
#
# Exported graph: one H-cycle pass (backbone only, puzzle_emb external)
#   Inputs : x (B, seq_len), puzzle_emb_row (B, emb_dim), z_H, z_L
#   Outputs: logits, z_H_new, z_L_new
import os, time

class TRMBackboneStep(nn.Module):
    """Single H-cycle backbone step. Puzzle embedding passed as float input."""
    def __init__(self, inner_model):
        super().__init__()
        self.m = inner_model

    def forward(self, x, puzzle_emb_row, z_H, z_L):
        from models.recursive_reasoning.trm import (
            TinyRecursiveReasoningModel_ACTV1Carry,
            TinyRecursiveReasoningModel_ACTV1InnerCarry,
        )
        # Patch puzzle_emb to return the pre-looked-up row
        orig = self.m.inner.puzzle_emb.forward
        cast_to = self.m.inner.puzzle_emb.cast_to
        def _injected(ids): return puzzle_emb_row.to(cast_to)
        self.m.inner.puzzle_emb.forward = _injected

        pids  = torch.zeros(x.shape[0], dtype=torch.int32, device=x.device)
        batch = {"inputs": x.to(torch.int32), "labels": x.to(torch.int32),
                 "puzzle_identifiers": pids}
        carry = TinyRecursiveReasoningModel_ACTV1Carry(
            inner_carry=TinyRecursiveReasoningModel_ACTV1InnerCarry(z_H=z_H, z_L=z_L),
            steps=torch.zeros(x.shape[0], dtype=torch.int32, device=x.device),
            halted=torch.zeros(x.shape[0], dtype=torch.bool, device=x.device),
            current_data=batch,
        )
        try:
            new_carry, outputs = self.m(carry, batch)
        finally:
            self.m.inner.puzzle_emb.forward = orig
        return outputs["logits"], new_carry.inner_carry.z_H, new_carry.inner_carry.z_L

if 'model' in dir():
    try:
        inner   = get_inner(model)
        dev     = str(DEVICE)
        seq_len = inner.config.seq_len
        emb_len = inner.inner.puzzle_emb_len
        hidden  = inner.config.hidden_size
        emb_dim = inner.inner.puzzle_emb.weights.shape[1]

        print(f"Model: seq={seq_len}, emb_len={emb_len}, hidden={hidden}, emb_dim={emb_dim}")

        wrapper = TRMBackboneStep(inner).eval().to(dev)

        # Example inputs on the same device as the model
        x_d   = torch.zeros(1, seq_len,              dtype=torch.int64,  device=dev)
        emb_d = torch.zeros(1, emb_dim,              dtype=torch.float32, device=dev)
        zH_d  = torch.zeros(1, seq_len + emb_len, hidden, dtype=torch.float32, device=dev)
        zL_d  = torch.zeros(1, seq_len + emb_len, hidden, dtype=torch.float32, device=dev)

        # Cast model to float32 for deterministic trace (bfloat16 has numerical noise)
        wrapper = wrapper.float()

        print("Tracing with torch.jit.trace...")
        t0 = time.perf_counter()
        with torch.no_grad():
            traced = torch.jit.trace(wrapper, (x_d, emb_d, zH_d, zL_d),
                                     strict=False)    # strict=False tolerates dict returns
        dt = time.perf_counter() - t0
        print(f"  Trace done in {dt:.1f}s")

        # Save
        ts_path = "trm_backbone_step.pt"
        traced.save(ts_path)
        size_mb = os.path.getsize(ts_path) / 1024 / 1024
        print(f"  Saved: {ts_path}  ({size_mb:.2f} MB)")

        # Verify round-trip
        loaded = torch.jit.load(ts_path).to(dev)
        with torch.no_grad():
            out_orig   = wrapper(x_d, emb_d, zH_d, zL_d)
            out_loaded = loaded(x_d, emb_d, zH_d, zL_d)
        diff = (out_orig[0] - out_loaded[0]).abs().max().item()
        print(f"  Round-trip max abs diff: {diff:.2e}  (expect ~0)")

        print(f"\nEdge deployment flow:")
        print(f"  1. Load trm_backbone_step.pt  with torch::jit::load() in LibTorch C++")
        print(f"  2. Load puzzle_emb.weights[puzzle_id] row ({emb_dim*4/1024:.1f} KB) from flash")
        print(f"  3. Run traced(x, emb_row, z_H, z_L) for n_sup_max steps")

    except Exception as e:
        import traceback; traceback.print_exc()
else:
    print("[SKIP] model not loaded.")



In [ ]:
# ── 14.2  Latency benchmark: traced model vs eager ────────────────────────────
import os, time

ts_path = "trm_backbone_step.pt"
if 'model' in dir() and os.path.exists(ts_path):
    try:
        inner   = get_inner(model)
        dev     = str(DEVICE)
        seq_len = inner.config.seq_len
        emb_len = inner.inner.puzzle_emb_len
        hidden  = inner.config.hidden_size
        emb_dim = inner.inner.puzzle_emb.weights.shape[1]

        x_d   = torch.zeros(1, seq_len,              dtype=torch.int64,  device=dev)
        emb_d = torch.zeros(1, emb_dim,              dtype=torch.float32, device=dev)
        zH_d  = torch.zeros(1, seq_len + emb_len, hidden, dtype=torch.float32, device=dev)
        zL_d  = torch.zeros(1, seq_len + emb_len, hidden, dtype=torch.float32, device=dev)

        loaded = torch.jit.load(ts_path).to(dev).eval()
        wrapper = TRMBackboneStep(inner).eval().to(dev).float()

        N_WARMUP, N_BENCH = 3, 20

        def bench(fn, label):
            # Warm-up
            for _ in range(N_WARMUP):
                with torch.no_grad(): fn()
            if dev == "cuda": torch.cuda.synchronize()
            t0 = time.perf_counter()
            for _ in range(N_BENCH):
                with torch.no_grad(): fn()
            if dev == "cuda": torch.cuda.synchronize()
            ms = (time.perf_counter() - t0) / N_BENCH * 1000
            print(f"  {label:<30}: {ms:6.2f} ms/step")
            return ms

        print(f"Latency benchmark ({N_BENCH} runs, device={dev}):")
        ms_eager  = bench(lambda: wrapper(x_d, emb_d, zH_d, zL_d), "Eager (float32)")
        ms_traced = bench(lambda: loaded(x_d, emb_d, zH_d, zL_d),  "TorchScript traced")
        print(f"\n  TorchScript speedup : {ms_eager/ms_traced:.2f}×")

        # Full inference estimate (n_sup_max steps)
        n_steps = 16
        print(f"\n  Full inference ({n_steps} H-cycle steps):")
        print(f"    Eager   : {ms_eager  * n_steps:.1f} ms/puzzle")
        print(f"    Traced  : {ms_traced * n_steps:.1f} ms/puzzle")

        # File size summary
        ts_mb = os.path.getsize(ts_path) / 1024 / 1024
        print(f"\nFile sizes:")
        print(f"  {ts_path:<35}: {ts_mb:.2f} MB  (backbone, FP32)")
        print(f"  puzzle_emb.weights (INT8, full table): "
              f"{inner.inner.puzzle_emb.weights.numel()/1024:.1f} KB")
        print(f"  puzzle_emb row (1 puzzle, FP32)     : {emb_dim*4/1024:.1f} KB")

    except Exception as e:
        import traceback; traceback.print_exc()
else:
    print(f"[SKIP] model not loaded or {ts_path} not found — run cell 14.1 first.")


---
## Section 15 — QAT with Proper Train / Val Split

The §12 QAT evaluated on the **same** data it trained on → loss 632→1.2 in 50 steps looks
like overfitting. This section re-runs QAT with an 80/20 split and evaluates on the held-out
val set to get an honest accuracy number.


In [ ]:
# ── 15.1  Build train / val loaders ──────────────────────────────────────────
import math
from torch.utils.data import Subset

if train_loader is not None:
    full_ds   = train_loader.dataset
    n_total   = len(full_ds)
    n_val     = max(1, math.floor(n_total * 0.20))
    n_train   = n_total - n_val

    # Deterministic split (no shuffle of indices so puzzle groups stay intact)
    all_idx   = list(range(n_total))
    train_idx = all_idx[:n_train]
    val_idx   = all_idx[n_train:]

    train_sub = Subset(full_ds, train_idx)
    val_sub   = Subset(full_ds, val_idx)

    BATCH = train_loader.batch_size
    qat_train_loader = torch.utils.data.DataLoader(
        train_sub, batch_size=BATCH, shuffle=True,
        collate_fn=train_loader.collate_fn if hasattr(train_loader, 'collate_fn') else None,
        drop_last=True,
    )
    qat_val_loader = torch.utils.data.DataLoader(
        val_sub, batch_size=BATCH, shuffle=False,
        collate_fn=train_loader.collate_fn if hasattr(train_loader, 'collate_fn') else None,
        drop_last=False,
    )
    print(f"Train split : {len(train_sub):,} samples")
    print(f"Val   split : {len(val_sub):,}  samples")
else:
    print("[SKIP] train_loader not available.")


In [ ]:
# # ── 15.2  QAT loop with val-set evaluation every 10 steps ────────────────────
# import copy, numpy as np

# def run_qat_validated(base_mdl, tr_loader, v_loader,
#                       n_steps=100, lr=1e-5, device="cuda",
#                       eval_every=10, max_eval_batches=20):
#     """
#     QAT fine-tune on tr_loader, evaluate on v_loader every eval_every steps.
#     Returns (trained_model, history_dict).
#     """
#     from models.layers import CastedLinear

#     # Build INT4 fake-quant copy
#     m = copy.deepcopy(base_mdl)
#     inner_m = get_inner(m)
#     replaced = 0
#     for name, mod in list(inner_m.named_modules()):
#         if isinstance(mod, (nn.Linear, CastedLinear)):
#             parent_name, child_name = name.rsplit(".", 1) if "." in name else ("", name)
#             parent = inner_m if parent_name == "" else dict(inner_m.named_modules())[parent_name]
#             bias = mod.bias if (hasattr(mod, 'bias') and mod.bias is not None) else None
#             fq = FakeQuantINT4(mod.weight, bias)
#             setattr(parent, child_name, fq)
#             replaced += 1
#     print(f"  Replaced {replaced} layers → FakeQuantINT4")

#     m = m.to(device).train()
#     optimizer = torch.optim.AdamW(m.parameters(), lr=lr)
#     history   = {"step": [], "train_loss": [], "val_exact": [], "val_cell": []}

#     tr_iter = iter(tr_loader)
#     for step in range(n_steps):
#         # ── training step ──────────────────────────────────────────────────
#         try:
#             batch = next(tr_iter)
#         except StopIteration:
#             tr_iter = iter(tr_loader)
#             batch = next(tr_iter)

#         if isinstance(batch, (list, tuple)):
#             inputs, labels, pids = batch
#         elif isinstance(batch, dict):
#             inputs = batch["inputs"]
#             labels = batch["labels"]
#             pids = batch["puzzle_identifiers"]
#         else:
#             raise TypeError("Unsupported batch format")

#         # Micro-batching to fit in 4GB VRAM
#         micro_batch_size = 16
#         num_micro_batches = max(1, len(inputs) // micro_batch_size)
        
#         optimizer.zero_grad()
#         accum_loss = 0.0
        
#         for mb_idx in range(num_micro_batches):
#             mb_start = mb_idx * micro_batch_size
#             mb_end = mb_start + micro_batch_size
            
#             mb_x = inputs[mb_start:mb_end].to(device)
#             mb_y = labels[mb_start:mb_end].to(device)
#             mb_pids = pids[mb_start:mb_end].to(device)
            
#             batch_d = {
#                 "inputs":             mb_x,
#                 "labels":             mb_y,
#                 "puzzle_identifiers": mb_pids,
#             }
#             carry = m.initial_carry(batch_d)
#             carry = cast_carry_to_device(carry, device)
            
#             mb_loss = torch.tensor(0.0, device=device)
#             for _ in range(4):
#                 carry, outputs = m(carry, batch_d)
#                 logits = outputs["logits"]
#                 loss   = F.cross_entropy(
#                     logits.reshape(-1, logits.size(-1)),
#                     mb_y.reshape(-1), ignore_index=-100)
#                 mb_loss = mb_loss + loss
#                 if carry.halted.all(): break

#             mb_loss_normalized = mb_loss / num_micro_batches
#             mb_loss_normalized.backward()
#             accum_loss += mb_loss.item()

#         torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
#         optimizer.step()

#         # ── val evaluation ─────────────────────────────────────────────────
#         if (step + 1) % eval_every == 0:
#             pexact, cacc, _, _ = evaluate_arc_per_puzzle(
#                 m, v_loader, device=device, n_sup_max=16,
#                 max_batches=max_eval_batches)
#             history["step"].append(step + 1)
#             history["train_loss"].append(accum_loss)
#             history["val_exact"].append(pexact)
#             history["val_cell"].append(cacc)
#             print(f"  Step {step+1:4d}/{n_steps} | "
#                   f"loss={accum_loss:.4f} | "
#                   f"val_exact={pexact:.4f} | val_cell={cacc:.4f}")
#             # Free memory
#             import gc
#             gc.collect()
#             torch.cuda.empty_cache()

#     m.eval()
#     return m, history

# # Clean up old training variables to free up GPU memory
# import gc
# for var in ['qat_model', 'qat_model_v', 'fused_model', 'cpu_model', 'wrapper_cpu']:
#     if var in globals():
#         try:
#             del globals()[var]
#         except Exception:
#             pass
# gc.collect()
# torch.cuda.empty_cache()

# # if 'model' in dir() and 'qat_train_loader' in dir() and 'FakeQuantINT4' in dir():
# print('QAT split fine-tuning commented out.')
# # [COMMENTED OUT FOR MODAL EXECUTION TO PREVENT SLOW EXPERIMENTS]
# #     print("Starting validated QAT (100 steps, lr=1e-5)...")
# #     qat_model_v, qat_history = run_qat_validated(
# #         model, qat_train_loader, qat_val_loader,
# #         n_steps=100, lr=1e-5, device=str(DEVICE),
# #         eval_every=10, max_eval_batches=20)
# # else:
# #     print("[SKIP] Prerequisites missing — run §12 cell first for FakeQuantINT4.")


In [ ]:
# ── 15.3  Plot train loss + val accuracy curve ────────────────────────────────
if 'qat_history' in dir() and qat_history["step"]:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(qat_history["step"], qat_history["train_loss"], "b-o", ms=4)
    ax1.set_xlabel("QAT Step"); ax1.set_ylabel("Train Loss")
    ax1.set_title("QAT Training Loss"); ax1.grid(True, alpha=0.3)

    ax2.plot(qat_history["step"], qat_history["val_exact"], "g-o", ms=4, label="Puzzle Exact")
    ax2.plot(qat_history["step"], qat_history["val_cell"],  "r-o", ms=4, label="Cell Acc")
    ax2.set_xlabel("QAT Step"); ax2.set_ylabel("Accuracy")
    ax2.set_title("Val Accuracy During QAT"); ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("qat_val_curve.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Saved: qat_val_curve.png")

    best_idx = max(range(len(qat_history["val_exact"])),
                   key=lambda i: qat_history["val_exact"][i])
    print(f"\nBest val result at step {qat_history['step'][best_idx]}:")
    print(f"  Puzzle exact : {qat_history['val_exact'][best_idx]:.4f}")
    print(f"  Cell acc     : {qat_history['val_cell'][best_idx]:.4f}")
else:
    print("[SKIP] No QAT history — run §15.2 first.")


---
## Section 16 — INT8 Backbone + Single-Puzzle Fused Artifact

Best accuracy/size combo identified in the analysis:
- **INT8 backbone** (bitsandbytes): 6.7 MB, preserves ~19-20% cell accuracy
- **Single-puzzle embedding**: 2 KB SRAM, full table on flash

This section fuses both into a single deployment-ready TorchScript `.pt` file
and measures accuracy on the held-out val set.


In [ ]:
# ── 16.1  Build fused INT8 + SinglePuzzle model ───────────────────────────────
import copy

if 'model' in dir() and 'INT8PuzzleEmbedding' in dir() and 'SinglePuzzleEmbedding' in dir():
    print("Building INT8-backbone + single-puzzle model...")

    # Step 1: Start from INT8 (bnb) backbone variant.
    # variants stores (model, device_str) tuples — unpack accordingly.
    int8_backbone, _dev = variants["INT8 (bnb)"]
    fused_model = copy.deepcopy(int8_backbone)

    # Step 2: Get the puzzle_emb from the *original* model's inner (int8 model
    # is already the unwrapped inner, but puzzle_emb lives on .inner.puzzle_emb).
    orig_inner  = get_inner(model)           # original FP32 model
    orig_emb    = orig_inner.inner.puzzle_emb  # the real embedding module

    # Step 3: Replace puzzle_emb on the INT8 model with SinglePuzzleEmbedding.
    pw      = orig_emb.weights.cpu().float()
    cast_to = orig_emb.cast_to
    sp_emb  = SinglePuzzleEmbedding(pw, puzzle_id=0, cast_to=cast_to)
    fused_model.inner.puzzle_emb = sp_emb

    fused_model = fused_model.to(DEVICE).eval()

    # Size accounting
    import io
    def state_dict_kb(m):
        buf = io.BytesIO()
        torch.save(m.state_dict(), buf)
        return buf.tell() / 1024

    backbone_kb  = state_dict_kb(fused_model)
    emb_full_kb  = orig_emb.weights.numel() * 1 / 1024  # INT8 table (1 byte/element)
    emb_row_kb   = orig_emb.weights.shape[1] * 4 / 1024 # 1 FP32 row in SRAM

    print(f"\nFused model breakdown:")
    print(f"  INT8 backbone .pt size : {backbone_kb:>10.1f} KB  ({backbone_kb/1024:.2f} MB)")
    print(f"  INT8 emb table (flash) : {emb_full_kb:>10.1f} KB  ({emb_full_kb/1024:.2f} MB)")
    print(f"  Active emb row (SRAM)  : {emb_row_kb:>10.2f} KB")
    print(f"\n  ✓ SRAM during inference: {backbone_kb + emb_row_kb:.1f} KB  ({(backbone_kb+emb_row_kb)/1024:.2f} MB)")
    print(f"  Fits 4MB SRAM?  {'✓' if backbone_kb + emb_row_kb < 4096 else '✗'}")
    print(f"  Fits 8MB SRAM?  {'✓' if backbone_kb + emb_row_kb < 8192 else '✗'}")
else:
    print("[SKIP] model or helpers not available — run §9 cells first.")


In [ ]:
# ── 16.2  Accuracy of fused model (val set, per-puzzle aggregation) ──────────
if 'fused_model' in dir() and 'qat_val_loader' in dir():
    loader_to_use = qat_val_loader
    label = "val"
elif 'fused_model' in dir() and train_loader is not None:
    loader_to_use = train_loader
    label = "train (no val split available)"
else:
    loader_to_use = None

if loader_to_use is not None:
    print(f"Evaluating fused INT8+SinglePuzzle model on {label} set...")
    fp32_exact, fp32_cell, fp32_ms, _ = evaluate_arc_per_puzzle(
        model, loader_to_use, device=str(DEVICE), n_sup_max=16, max_batches=30)
    fused_exact, fused_cell, fused_ms, fused_n = evaluate_arc_per_puzzle(
        fused_model, loader_to_use, device=str(DEVICE), n_sup_max=16, max_batches=30)

    print(f"\n{'Variant':<30} {'Puzzle Exact':>12} {'Cell Acc':>10} {'ms/puzzle':>10}")
    print("-" * 65)
    print(f"  {'FP32 (bf16) baseline':<28} {fp32_exact:>12.4f} {fp32_cell:>10.4f} {fp32_ms:>10.2f}")
    print(f"  {'INT8 + Single-Puzzle (fused)':<28} {fused_exact:>12.4f} {fused_cell:>10.4f} {fused_ms:>10.2f}")
    delta_exact = fused_exact - fp32_exact
    delta_cell  = fused_cell  - fp32_cell
    sign_e = '+' if delta_exact >= 0 else ''
    sign_c = '+' if delta_cell  >= 0 else ''
    print(f"  {'Δ vs FP32':<28} {sign_e}{delta_exact:>12.4f} {sign_c}{delta_cell:>10.4f}")
else:
    print("[SKIP] fused_model not available.")


In [ ]:
# ── 16.3  Save fused model (state dict + embedding table) ────────────────────
# Note: bitsandbytes INT8 ops are NOT torch.jit.trace-able (custom CUDA kernels).
# For edge deployment we save:
#   (a) state dict .pt  → load with torch.load() + model skeleton
#   (b) puzzle_emb INT8 binary → seek-and-load per puzzle at runtime

import os, io

if 'fused_model' in dir():
    # (a) Save state dict
    sd_path = "trm_int8_singlepuzzle_state.pt"
    torch.save(fused_model.state_dict(), sd_path)
    sd_mb = os.path.getsize(sd_path) / 1024 / 1024
    print(f"Saved state dict : {sd_path}  ({sd_mb:.2f} MB)")

    # (b) Save full INT8 embedding table as a raw binary (simulate flash storage)
    orig_inner = get_inner(model)
    orig_emb_weights = orig_inner.inner.puzzle_emb.weights   # (N, D) float32
    emb_table_path = "puzzle_emb_fp32_table.pt"
    torch.save(orig_emb_weights.cpu(), emb_table_path)
    emb_mb = os.path.getsize(emb_table_path) / 1024 / 1024
    print(f"Saved emb table  : {emb_table_path}  ({emb_mb:.2f} MB)  ← flash storage")

    # (c) Size of one embedding row in SRAM
    emb_dim   = orig_emb_weights.shape[1]
    row_kb    = emb_dim * 4 / 1024
    backbone_mb = sd_mb

    print(f"\nDeployment package breakdown:")
    print(f"  Backbone state dict  : {backbone_mb:.2f} MB  (SRAM — INT8 weights)")
    print(f"  Embedding table      : {emb_mb:.2f} MB  (flash)")
    print(f"  Active emb row       : {row_kb:.2f} KB  (SRAM per puzzle)")
    print(f"\n  Note: TorchScript trace is not supported for bitsandbytes INT8 models.")
    print(f"  Use torch.load(state_dict) + model skeleton for deployment instead.")
    print(f"  For true edge export, convert backbone to PyTorch native INT8 first.")
else:
    print("[SKIP] fused_model not available — run §16.1 first.")


---
## Section 17 — Simulated Edge Deployment Profile

No physical edge hardware available, so we simulate the deployment environment:

- **CPU-only latency**: move the INT8+SinglePuzzle model to CPU and benchmark there
  (representative of ARM Cortex-M55 / ESP32-S3 class devices)
- **Peak SRAM usage**: track memory high-watermark during inference with `tracemalloc`
- **FLOP count**: estimate via `torch.profiler`
- **Estimated power**: rough estimate using published Cortex-M55 efficiency figures

*Note: CPU latency on a server CPU is not the same as on-device, but it gives a useful
order-of-magnitude estimate and validates that inference is architecturally feasible.*


In [ ]:
# ── 17.1  CPU-only latency benchmark ─────────────────────────────────────────
import time, gc, copy

if 'fused_model' in dir():
    print("Moving fused model to CPU for edge-device latency simulation...")

    # Use the FP32 baseline model for CPU benchmark (bnb INT8 has CUDA-only kernels)
    # This gives us FLOPs / latency profile representative of the architecture
    cpu_model = copy.deepcopy(get_inner(model)).cpu().float().eval()

    inner_cpu = cpu_model   # already unwrapped
    seq_len   = inner_cpu.config.seq_len
    emb_len   = inner_cpu.inner.puzzle_emb_len
    hidden    = inner_cpu.config.hidden_size
    emb_dim   = inner_cpu.inner.puzzle_emb.weights.shape[1]

    x_cpu   = torch.zeros(1, seq_len,               dtype=torch.int64)
    emb_cpu = torch.zeros(1, emb_dim,               dtype=torch.float32)
    zH_cpu  = torch.zeros(1, seq_len + emb_len, hidden, dtype=torch.float32)
    zL_cpu  = torch.zeros(1, seq_len + emb_len, hidden, dtype=torch.float32)

    wrapper_cpu = TRMBackboneStep(inner_cpu).eval().cpu().float()

    N_WARMUP, N_BENCH = 1, 3   # CPU is slow, keep small
    for _ in range(N_WARMUP):
        with torch.no_grad():
            wrapper_cpu(x_cpu, emb_cpu, zH_cpu, zL_cpu)

    t0 = time.perf_counter()
    for _ in range(N_BENCH):
        with torch.no_grad():
            wrapper_cpu(x_cpu, emb_cpu, zH_cpu, zL_cpu)
    ms_per_step = (time.perf_counter() - t0) / N_BENCH * 1000

    n_sup_max = 16
    ms_per_puzzle = ms_per_step * n_sup_max

    print(f"\nCPU latency (server CPU, FP32, 1 H-cycle step) : {ms_per_step:.1f} ms/step")
    print(f"Full inference ({n_sup_max} steps)                  : {ms_per_puzzle:.0f} ms/puzzle")
    print(f"\nEstimated on-device scaling (order-of-magnitude):")
    for label, factor in [("Cortex-A55 (mobile, ~1 TFLOPS)", 3),
                           ("Cortex-M55 (MCU, ~128 GFLOPS)",  20),
                           ("ESP32-S3 LX7 (~40 GFLOPS)",      50)]:
        est_s = ms_per_puzzle * factor / 1000
        print(f"  {label:<40}: ~{est_s:.1f}s / puzzle")
else:
    print("[SKIP] fused_model not available.")


In [ ]:
# ── 17.2  Peak SRAM usage during inference (tracemalloc) ─────────────────────
import tracemalloc

if 'wrapper_cpu' in dir():
    gc.collect()
    tracemalloc.start()
    snapshot_before = tracemalloc.take_snapshot()

    with torch.no_grad():
        for _ in range(3):
            wrapper_cpu(x_cpu, emb_cpu, zH_cpu, zL_cpu)

    snapshot_after = tracemalloc.take_snapshot()
    tracemalloc.stop()

    # Peak delta
    stats = snapshot_after.compare_to(snapshot_before, "lineno")
    peak_kb = sum(s.size_diff for s in stats if s.size_diff > 0) / 1024
    top_stats = sorted(stats, key=lambda s: s.size_diff, reverse=True)[:5]

    print(f"Peak SRAM delta during inference : {peak_kb:,.0f} KB  ({peak_kb/1024:.2f} MB)")
    print(f"\nTop allocations:")
    for s in top_stats:
        if s.size_diff > 0:
            print(f"  {s.size_diff/1024:>8.1f} KB  {str(s.traceback[0])}")
else:
    print("[SKIP] Run §17.1 first.")


In [ ]:
# ── 17.3  FLOP estimate via torch.profiler ────────────────────────────────────
try:
    from torch.profiler import profile, ProfilerActivity, record_function

    if 'wrapper_cpu' in dir():
        with profile(activities=[ProfilerActivity.CPU],
                     record_shapes=True,
                     with_flops=True) as prof:
            with record_function("trm_step"):
                with torch.no_grad():
                    wrapper_cpu(x_cpu, emb_cpu, zH_cpu, zL_cpu)

        total_flops = sum(e.flops for e in prof.key_averages() if e.flops)
        print(f"Estimated FLOPs per H-cycle step : {total_flops/1e6:.1f} MFLOPs")
        n_sup_max = 10
        print(f"Full inference ({n_sup_max} steps)        : {total_flops*n_sup_max/1e9:.2f} GFLOPs")

        print(f"\nPower estimate (Cortex-M55 @ ~1 GFLOPS/s, ~1 mW/GFLOP):")
        gflops_total = total_flops * 10 / 1e9
        print(f"  FLOPs                  : {gflops_total:.3f} GFLOPs/puzzle")
        print(f"  Est. energy            : ~{gflops_total:.2f} mJ/puzzle")
        print(f"  Puzzles per mAh (3.3V) : ~{3600*3.3/1000/max(gflops_total,0.001):.0f}")
    else:
        print("[SKIP] Run §17.1 first.")
except Exception as e:
    print(f"[WARN] torch.profiler FLOP counting: {e}")
    print("FLOPs unavailable on this PyTorch build — skipping.")


In [ ]:
# ── 17.4  Deployment summary table ───────────────────────────────────────────
if 'fused_exact' in dir() and 'fused_cell' in dir():
    import pandas as pd

    # Get baseline results from results dict dynamically
    fp32_exact = results.get("FP32 (bf16)", {}).get("exact_acc", results.get("FP16", {}).get("exact_acc", 0.8680))
    fp32_cell  = results.get("FP32 (bf16)", {}).get("cell_acc",  results.get("FP16", {}).get("cell_acc",  0.9952))

    int8_exact = results.get("INT8 (bnb)", {}).get("exact_acc", 0.8700)
    int8_cell  = results.get("INT8 (bnb)", {}).get("cell_acc",  0.9953)

    int4_exact = results.get("INT4 (fake)", {}).get("exact_acc", 0.8640)
    int4_cell  = results.get("INT4 (fake)", {}).get("cell_acc",  0.9950)

    f_exact = globals().get("fused_exact", int8_exact)
    f_cell  = globals().get("fused_cell", int8_cell)

    # Dynamic parameter/buffer sizes
    inner = get_inner(model)
    param_numel = sum(p.numel() for p in inner.parameters())
    N, D = inner.inner.puzzle_emb.weights.shape

    backbone_fp32_mb = (param_numel * 4) / (1024 * 1024)
    backbone_int8_mb = (param_numel * 1) / (1024 * 1024)
    backbone_int4_mb = (param_numel * 0.5) / (1024 * 1024)

    puzzle_fp32_mb = (N * D * 4) / (1024 * 1024)
    single_row_mb  = (D * 4) / (1024 * 1024)

    total_fp32_mb  = backbone_fp32_mb + puzzle_fp32_mb
    total_int8_mb  = backbone_int8_mb + puzzle_fp32_mb
    total_int4_mb  = backbone_int4_mb + single_row_mb
    total_fused_mb = backbone_int8_mb + single_row_mb

    fits_4mb = lambda mb: "✓" if mb < 4.0 else "✗"
    fits_8mb = lambda mb: "✓" if mb < 8.0 else "✗"

    rows = [
        {"Config":           "FP32 (baseline)",
         "Backbone MB":      backbone_fp32_mb,
         "Total deploy MB":  total_fp32_mb,
         "Puzzle Exact":     fp32_exact,
         "Cell Acc":         fp32_cell,
         "Fits 4MB":         fits_4mb(total_fp32_mb),
         "Fits 8MB":         fits_8mb(total_fp32_mb)},
        {"Config":           "INT8 (bnb)",
         "Backbone MB":      backbone_int8_mb,
         "Total deploy MB":  total_int8_mb,
         "Puzzle Exact":     int8_exact,
         "Cell Acc":         int8_cell,
         "Fits 4MB":         fits_4mb(total_int8_mb),
         "Fits 8MB":         fits_8mb(total_int8_mb)},
        {"Config":           "INT4 (calibrated)",
         "Backbone MB":      backbone_int4_mb,
         "Total deploy MB":  total_int4_mb,
         "Puzzle Exact":     int4_exact,
         "Cell Acc":         int4_cell,
         "Fits 4MB":         fits_4mb(total_int4_mb),
         "Fits 8MB":         fits_8mb(total_int4_mb)},
        {"Config":           "INT8 + Single-Puzzle ★",
         "Backbone MB":      backbone_int8_mb,
         "Total deploy MB":  total_fused_mb,
         "Puzzle Exact":     f_exact,
         "Cell Acc":         f_cell,
         "Fits 4MB":         fits_4mb(total_fused_mb),
         "Fits 8MB":         fits_8mb(total_fused_mb)},
    ]

    df = pd.DataFrame(rows).set_index("Config")
    df["Backbone MB"] = df["Backbone MB"].map("{:.2f}".format)
    df["Total deploy MB"] = df["Total deploy MB"].map("{:.2f}".format)
    df["Puzzle Exact"] = df["Puzzle Exact"].map("{:.4f}".format)
    df["Cell Acc"] = df["Cell Acc"].map("{:.4f}".format)
    print("\n=== Final Deployment Scorecard ===")
    print(df.to_string())
    print("\n★ = recommended target for 8MB SRAM devices")
else:
    print("[SKIP] Run §16.2 first.")


---
## Summary of New Results (§9–§14)

| Section | What we did | Key finding |
|---|---|---|
| §9 | Puzzle embedding compression | INT8 emb: ~15 MB; SVD r=16: may fit 4MB; single-puzzle: backbone-only |
| §10 | Fixed per-puzzle evaluation | Numbers now comparable to 68.89% submission baseline |
| §11 | Calibrated INT4 (per-channel) | Carry similarity stays near FP32; no collapse |
| §12 | QAT fine-tuning (50 steps) | Recovers accuracy gap from INT4 quantization |
| §13 | Structured pruning (25%/50%) | Accuracy vs. sparsity trade-off measured |
| §14 | ONNX export | Single-step graph exported; validated with onnxruntime |

### Remaining path to true edge deployment
1. Combine INT8-emb + INT4-backbone + QAT → single deployment artifact
2. Export ONNX without embedded puzzle buffers (load from flash)
3. Benchmark on actual ARM Cortex-M / ESP32-S3 hardware with ONNX Runtime or TFLite


---
## Section 15 — QAT with Proper Train / Val Split

The §12 QAT evaluated on the **same** data it trained on → loss 632→1.2 in 50 steps looks
like overfitting. This section re-runs QAT with an 80/20 split and evaluates on the held-out
val set to get an honest accuracy number.


In [ ]:
# # ── Utility: fix H_init / L_init device mismatch (permanent patch) ───────────
# # There are TWO trm.py files on this Modal volume:
# #   1. TinyRecursiveModels/trm.py           ← what the loaded model uses
# #   2. models/recursive_reasoning/trm.py    ← what our import statements find
# #
# # We MUST patch the actual class the model uses, not the imported one.
# # Solution: get the class from the live model object via type().

# if "model" in dir():
#     TRM_Inner = type(model.inner)   # ← exact class, whatever module it came from

#     if not getattr(TRM_Inner, "_hinit_patched", False):
#         _orig_reset = TRM_Inner.reset_carry

#         def _patched_reset(self, reset_flag, carry):
#             dev = reset_flag.device
#             if self.H_init.device != dev:
#                 self.H_init = self.H_init.to(dev)
#             if self.L_init.device != dev:
#                 self.L_init = self.L_init.to(dev)
#             return _orig_reset(self, reset_flag, carry)

#         TRM_Inner.reset_carry = _patched_reset
#         TRM_Inner._hinit_patched = True
#         print(f"✓ Patched {TRM_Inner.__module__}.{TRM_Inner.__name__}.reset_carry")
#         print(f"  H_init device before: {model.inner.H_init.device}")
#     else:
#         print("✓ reset_carry already patched.")
# else:
#     print("[SKIP] model not available — run model-loading cells first.")


---
## Section 18 — 1-Cycle Inference: 4× Compute Reduction

INT8 peaks at H_cycles=1 (cell acc 20.9%) vs FP32 peaking at H_cycles=4 (21.9%).
This section confirms that result on the full test set and quantifies the compute saving.


In [ ]:
# ── 18.1  Evaluate all variants at n_sup_max = 1, 2, 4, 8, 16 (H_cycles = default vs 1) ──
# Use test_loader (same as §10) so numbers are directly comparable.

def fix_hinit_device(m, device):
    """register_buffer is the only reliable way to move nn.Buffer to device."""
    for sub in m.modules():
        for attr in ("H_init", "L_init"):
            try:
                val = getattr(sub, attr)
                if isinstance(val, torch.Tensor) and val.device.type != device.split(":")[0]:
                    sub.register_buffer(attr, val.to(device))
            except AttributeError:
                pass
    return m

if "variants" in dir() and "test_loader" in dir():
    FLOPS_PER_STEP = 25_631  # MFLOPs from §17.3

    N_STEPS_SWEEP = [1, 2, 4, 8, 16]

    for vname, (vm, dev) in variants.items():
        try:
            fix_hinit_device(vm, dev)
            
            # Fetch model default H_cycles
            inner = get_inner(vm)
            default_h = inner.config.H_cycles
            
            print("=" * 86)
            print(f"VARIANT: {vname:<25} (device: {dev})")
            print("=" * 86)
            print(f"{'H_cycles':<10} {'Metric':<10} " + " ".join(f"{f'n={n}':>10}" for n in N_STEPS_SWEEP))
            print("-" * 86)
            
            # 1. Evaluate with default H_cycles
            results_def = {"exact": [], "cell": [], "ms": [], "flops": []}
            for n_sup in N_STEPS_SWEEP:
                exact, cell, ms, _ = evaluate_arc_per_puzzle(
                    vm, test_loader, device=dev, n_sup_max=n_sup, max_batches=None, fast_mode=True
                )
                results_def["exact"].append(exact)
                results_def["cell"].append(cell)
                results_def["ms"].append(ms)
                results_def["flops"].append(n_sup * FLOPS_PER_STEP / 1000)
                
            print(f"{f'{default_h} (Def)':<10} {'Exact':<10} " + " ".join(f"{val:>10.4f}" for val in results_def["exact"]))
            print(f"{'':<10} {'Cell':<10} " + " ".join(f"{val:>10.4f}" for val in results_def["cell"]))
            print(f"{'':<10} {'Latency':<10} " + " ".join(f"{val:>8.1f} ms" for val in results_def["ms"]))
            print(f"{'':<10} {'GFLOPs':<10} " + " ".join(f"{val:>10.1f}" for val in results_def["flops"]))
            print("-" * 86)
            
            # 2. Evaluate with H_cycles overridden to 1
            orig_h = inner.config.H_cycles
            orig_ih = inner.inner.config.H_cycles
            
            results_h1 = {"exact": [], "cell": [], "ms": [], "flops": []}
            try:
                inner.config.H_cycles = 1
                inner.inner.config.H_cycles = 1
                
                # Scale FLOPs per step down by default_h since we do 1 cycle instead of default_h
                flops_per_step_h1 = 8_544
                
                for n_sup in N_STEPS_SWEEP:
                    exact, cell, ms, _ = evaluate_arc_per_puzzle(
                        vm, test_loader, device=dev, n_sup_max=n_sup, max_batches=None, fast_mode=True
                    )
                    results_h1["exact"].append(exact)
                    results_h1["cell"].append(cell)
                    results_h1["ms"].append(ms)
                    results_h1["flops"].append(n_sup * flops_per_step_h1 / 1000)
            finally:
                # Restore original H_cycles configuration
                inner.config.H_cycles = orig_h
                inner.inner.config.H_cycles = orig_ih
                
            print(f"{'1':<10} {'Exact':<10} " + " ".join(f"{val:>10.4f}" for val in results_h1["exact"]))
            print(f"{'':<10} {'Cell':<10} " + " ".join(f"{val:>10.4f}" for val in results_h1["cell"]))
            print(f"{'':<10} {'Latency':<10} " + " ".join(f"{val:>8.1f} ms" for val in results_h1["ms"]))
            print(f"{'':<10} {'GFLOPs':<10} " + " ".join(f"{val:>10.1f}" for val in results_h1["flops"]))
            print("=" * 86 + "\n")
            
        except Exception as ex:
            print(f"  {vname:<18} ERROR: {ex}\n")
else:
    print("[SKIP] variants or test_loader not available.")


---
## Section 19 — Sequence Length Ablation

The 81-token sequence dominates FLOPs (attention is O(n²)).
We test accuracy when the model only sees the first N tokens (rest zero-padded).
This tells us whether a future model trained with shorter sequences would retain accuracy.

FLOPs scale as (N/81)² × current FLOPs.


In [ ]:
# ── 19.1  Zero-pad truncation at various context lengths ────────────────────
@torch.no_grad()
def evaluate_truncated(mdl, loader, trunc_len, device="cuda", n_sup_max=1, max_batches=None):
    """Evaluate model with tokens beyond trunc_len zeroed using the optimized evaluate_arc_per_puzzle."""
    res = evaluate_arc_per_puzzle(
        mdl, loader, device=device, n_sup_max=n_sup_max,
        max_batches=max_batches, return_pass2=False, fast_mode=True, trunc_len=trunc_len
    )
    # evaluate_arc_per_puzzle returns: pass_1_acc, cell_acc, ms, npuzz
    p1, cell, ms, npuzz = res
    return p1, cell

if "model" in dir() and "test_loader" in dir():
    inner_model = get_inner(model)
    seq_full = inner_model.config.seq_len
    emb_len = getattr(inner_model, "puzzle_emb_len", 16)
    print(f"Full seq_len = {seq_full}  (FLOPs ∝ seq_len due to MLP architecture)")
    print(f"{'Context':>10} {'% of full':>10} {'MLP FLOPs':>12} {'Puzzle Exact':>14} {'Cell Acc':>10}")
    print("-" * 62)

    for trunc in [seq_full, seq_full // 2, seq_full // 4, 16]:
        # Compute exact ratio including puzzle embedding length
        ratio = (trunc + emb_len) / (seq_full + emb_len)
        p, c  = evaluate_truncated(model, test_loader, trunc_len=trunc,
                                   device=str(DEVICE), n_sup_max=1, max_batches=None)
        flop_label = f"~{ratio * 25.631:.2f} GFLOPs"
        print(f"  {trunc:>8}  {trunc/seq_full*100:>9.0f}%  {flop_label:>12}  {p:>14.4f}  {c:>10.4f}")
else:
    print("[SKIP] model or test_loader not available.")


---
## Section 20 — Better QAT: Starting from Calibrated INT4

§15 showed QAT from naive INT4 memorises tokens without solving puzzles.
The calibrated INT4 (§11) already has carry fidelity 0.975 ≈ FP32.
Starting QAT from there with a lower LR should recover genuine reasoning accuracy.


In [ ]:
# # ── 20.1  QAT from calibrated INT4 (lr=1e-6, 300 steps) ────────────────────
# # Rebuild calibrated INT4 fresh (deepcopy fails on non-leaf bfloat16 tensors).
# # monkey-patch in the utility cell above handles H_init device automatically.

# if "quantize_int4_calibrated" in dir() and "model" in dir() and "train_loader" in dir():
#     print("Building fresh calibrated INT4 model for QAT...")
#     qat20_base = quantize_int4_calibrated(model)
#     if torch.cuda.is_available():
#         qat20_base = qat20_base.cuda()

#     qat20_base.train()
#     optimizer = torch.optim.AdamW(qat20_base.parameters(), lr=1e-6)
#     tr_iter   = iter(train_loader)
#     N_STEPS   = 300

#     for step in range(N_STEPS):
#         try:
#             batch = next(tr_iter)
#         except StopIteration:
#             tr_iter = iter(train_loader)
#             batch   = next(tr_iter)

#         inputs, labels, pids = batch
        
#         # Micro-batching to fit in 4GB VRAM
#         micro_batch_size = 16
#         num_micro_batches = max(1, len(inputs) // micro_batch_size)
        
#         optimizer.zero_grad()
#         accum_loss = 0.0
        
#         for mb_idx in range(num_micro_batches):
#             mb_start = mb_idx * micro_batch_size
#             mb_end = mb_start + micro_batch_size
            
#             mb_x = inputs[mb_start:mb_end].to(DEVICE)
#             mb_y = labels[mb_start:mb_end].to(DEVICE)
#             mb_pids = pids[mb_start:mb_end].to(DEVICE)
            
#             batch_d = {
#                 "inputs":             mb_x,
#                 "labels":             mb_y,
#                 "puzzle_identifiers": mb_pids,
#             }
#             carry      = qat20_base.initial_carry(batch_d)
#             carry      = cast_carry_to_device(carry, str(DEVICE))
            
#             mb_loss = torch.tensor(0., device=str(DEVICE))
#             for _ in range(4):
#                 carry, out = qat20_base(carry, batch_d)
#                 loss       = F.cross_entropy(
#                     out["logits"].reshape(-1, out["logits"].size(-1)),
#                     mb_y.reshape(-1), ignore_index=-100)
#                 mb_loss = mb_loss + loss
#                 if carry.halted.all():
#                     break
            
#             mb_loss_normalized = mb_loss / num_micro_batches
#             mb_loss_normalized.backward()
#             accum_loss += mb_loss.item()
            
#         torch.nn.utils.clip_grad_norm_(qat20_base.parameters(), 1.0)
#         optimizer.step()
        
#         if (step + 1) % 50 == 0:
#             print(f"  Step {step+1:4d}/{N_STEPS} | loss={accum_loss:.4f}")
#             # Free memory
#             import gc
#             gc.collect()
#             torch.cuda.empty_cache()

#     qat20_base.eval()
#     print("Evaluating on held-out test set...")
#     p, c, ms, n = evaluate_arc_per_puzzle(
#         qat20_base, test_loader, device=str(DEVICE), n_sup_max=1, max_batches=None)
#     print(f"  Calibrated INT4 + QAT (300 steps, lr=1e-6):")
#     print(f"    Puzzle exact : {p:.4f}")
#     print(f"    Cell acc     : {c:.4f}")
#     print(f"    SRAM         : ~3.3 MB backbone + 2KB emb row  (fits 4MB)")
# else:
#     print("[SKIP] quantize_int4_calibrated not defined — run §11 first.")


---
## Section 21 — Knowledge Distillation: Training a Smaller Student

We train a smaller TRM (hidden_size=256, 2 L-layers) using the FP32 model's output
logits as soft targets (KL divergence). This teaches the small model to mimic the
*distribution* of answers, not just the argmax — generalises better than hard labels.

Target: ~4× fewer parameters → ~4× fewer FLOPs.


In [ ]:
# ── 21.1  Instantiate student TRM and train with KL distillation ─────────────
# Student is built fresh with .to(DEVICE). monkey-patch (utility cell) handles H_init.
from models.recursive_reasoning.trm import TinyRecursiveReasoningModel_ACTV1
import os

if "model" in dir() and "train_loader" in dir():
    teacher_cfg      = get_inner(model).config
    student_cfg_dict = teacher_cfg.model_dump()
    student_cfg_dict.update({"hidden_size": 256, "num_heads": 4,
                              "L_layers": 1, "H_cycles": 1})

    student = TinyRecursiveReasoningModel_ACTV1(student_cfg_dict).to(DEVICE)
    s_params = sum(p.numel() for p in student.parameters())
    t_params = sum(p.numel() for p in get_inner(model).parameters())
    print(f"Teacher params : {t_params:,}")
    print(f"Student params : {s_params:,}  ({s_params/t_params:.2f}× teacher)")

    teacher = get_inner(model).eval()
    # We keep the learning rate at 3e-4 or slightly higher (e.g. 5e-4) to ensure stability.
    # Note: The KL loss is numerically large because PyTorch's 'batchmean' reduction sums over
    # the sequence length (81) and vocabulary dimension (11) before dividing only by batch size.
    # This does not mean the gradients are small, so a typical LR (3e-4 to 5e-4) is appropriate.
    optim   = torch.optim.AdamW(student.parameters(), lr=5e-4)
    T       = 4.0
    tr_iter = iter(train_loader)

    # Setup checkpoint directory
    os.makedirs("checkpoints", exist_ok=True)
    checkpoint_path = "checkpoints/student_distill_latest.pt"
    start_step = 0

    # Resume from checkpoint if it exists
    if os.path.exists(checkpoint_path):
        print(f"Resuming training from checkpoint: {checkpoint_path}")
        ckpt = torch.load(checkpoint_path, map_location=DEVICE)
        student.load_state_dict(ckpt["model_state_dict"])
        optim.load_state_dict(ckpt["optimizer_state_dict"])
        start_step = ckpt["step"]
        print(f"Resumed at step {start_step}")

    total_steps = 1000
    student.train()
    
    for step in range(start_step, total_steps):
        try:
            batch = next(tr_iter)
        except StopIteration:
            tr_iter = iter(train_loader)
            batch   = next(tr_iter)

        inputs, labels, pids = batch
        
        # Micro-batching to fit in 4GB VRAM
        micro_batch_size = 64
        num_micro_batches = max(1, len(inputs) // micro_batch_size)
        
        optim.zero_grad()
        accum_kl = 0.0
        accum_hard = 0.0
        
        for mb_idx in range(num_micro_batches):
            mb_start = mb_idx * micro_batch_size
            mb_end = mb_start + micro_batch_size
            
            mb_x = inputs[mb_start:mb_end].to(DEVICE)
            mb_y = labels[mb_start:mb_end].to(DEVICE)
            mb_pids = pids[mb_start:mb_end].to(DEVICE)
            
            mb_batch_d = {
                "inputs": mb_x,
                "labels": mb_y,
                "puzzle_identifiers": mb_pids
            }
            
            # Teacher forward pass (no_grad)
            with torch.no_grad():
                tc = teacher.initial_carry(mb_batch_d)
                tc = cast_carry_to_device(tc, DEVICE)
                for _ in range(4):
                    tc, t_out = teacher(tc, mb_batch_d)
                    if tc.halted.all(): break
                soft = (t_out["logits"] / T).log_softmax(-1)
                
            # Student forward pass
            sc = student.initial_carry(mb_batch_d)
            sc = cast_carry_to_device(sc, DEVICE)
            for _ in range(2):
                sc, s_out = student(sc, mb_batch_d)
                if sc.halted.all(): break
                
            kl   = F.kl_div((s_out["logits"]/T).log_softmax(-1), soft.exp(),
                             reduction="batchmean") * T**2
            hard = F.cross_entropy(s_out["logits"].reshape(-1, s_out["logits"].size(-1)),
                                   mb_y.reshape(-1), ignore_index=-100)
            
            mb_loss = 0.7 * kl + 0.3 * hard
            mb_loss_normalized = mb_loss / num_micro_batches
            mb_loss_normalized.backward()
            
            accum_kl += kl.item()
            accum_hard += hard.item()
            
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        optim.step()
        
        if (step + 1) % 40 == 0:
            print(f"  Step {step+1:4d}/{total_steps} | kl={accum_kl/num_micro_batches:.4f} | hard={accum_hard/num_micro_batches:.4f}")
            # Free memory
            import gc
            gc.collect()
            torch.cuda.empty_cache()
            
        # Save checkpoint periodically
        if (step + 1) % 200 == 0:
            checkpoint_data = {
                "step": step + 1,
                "model_state_dict": student.state_dict(),
                "optimizer_state_dict": optim.state_dict(),
                "kl_loss": accum_kl / num_micro_batches,
                "hard_loss": accum_hard / num_micro_batches,
            }
            # Save both a step-specific checkpoint and the latest pointer
            torch.save(checkpoint_data, f"checkpoints/student_distill_step_{step+1}.pt")
            torch.save(checkpoint_data, checkpoint_path)
            print(f"  [Checkpoint] Saved checkpoint at step {step+1} to {checkpoint_path}")

    student.eval()
    print("Evaluating student on test set...")
    p, c, ms, n = evaluate_arc_per_puzzle(
        student, test_loader, device=str(DEVICE), n_sup_max=1, max_batches=None)
    est = 25.631 * (256/512)**2
    print(f"  Student (hidden=256, 1L, 1-cycle): exact={p:.4f}  cell={c:.4f}  ~{est:.0f} GFLOPs")


---
## Section 22 — Linear Attention Approximation

Standard attention is O(n²) in sequence length.
We monkey-patch the `Attention.forward()` with a linear kernel:
  score(q,k) = (elu(q)+1) · (elu(k)+1)ᵀ   → O(n·d) instead of O(n²)

This is a zero-shot swap — no retraining. We test accuracy to see how much is lost,
and measure the theoretical FLOPs reduction.


In [ ]:
# ── 22.1  Monkey-patch softmax attention → linear attention ──────────────────
import copy
from models.layers import Attention
import einops

if "model" in dir() and "test_loader" in dir():
    inner_model = get_inner(model)
    has_attn = any(isinstance(mod, Attention) for mod in inner_model.modules())
    if not has_attn:
        print("The Sudoku Extreme model uses the TRM-MLP architecture (mlp_t = True), which has no self-attention layers.")
        print("Linear attention approximation is not applicable. Skipping.")
        # Define dummy variables to prevent NameError downstream
        lin_model = model
        p_lin, c_lin = 0.0, 0.0
        p_fp32, c_fp32 = 0.0, 0.0
    else:
        # Deep-copy the FP32 model and swap all Attention.forward
        lin_model = copy.deepcopy(get_inner(model)).to(DEVICE).eval()
        patched = 0
        for name, mod in lin_model.named_modules():
            if isinstance(mod, Attention):
                import types
                mod.forward = types.MethodType(linear_attention_forward, mod)
                patched += 1
        print(f"Patched {patched} Attention modules → linear O(n) attention")

        n_tokens = get_inner(model).config.seq_len
        flop_ratio = n_tokens / (get_inner(model).inner.L_level.layers[0].self_attn.head_dim)
        print(f"Theoretical compute reduction: ~{flop_ratio:.0f}× in attention layers")

        print("\nEvaluating linear attention model on test set (n_sup_max=1)...")
        p_lin, c_lin, ms_lin, _ = evaluate_arc_per_puzzle(
            lin_model, test_loader, device=str(DEVICE), n_sup_max=1, max_batches=None)

        print("\nEvaluating softmax attention (FP32 baseline, n_sup_max=1) for comparison...")
        p_fp32, c_fp32, ms_fp32, _ = evaluate_arc_per_puzzle(
            model, test_loader, device=str(DEVICE), n_sup_max=1, max_batches=None)

        print(f"\n{'Model':<30} {'Puzzle Exact':>13} {'Cell Acc':>10}")
        print("-" * 56)
        print(f"  {'FP32 softmax attn':<28} {p_fp32:>13.4f} {c_fp32:>10.4f}")
        print(f"  {'Linear attn (ELU)':<28} {p_lin:>13.4f} {c_lin:>10.4f}")
        print(f"\nAccuracy cost of linear swap: {(c_fp32-c_lin)*100:+.2f}pp cell accuracy")
        print(f"Note: linear attn needs retraining to recover accuracy — this is a zero-shot test.")
else:
    print("[SKIP] model or test_loader not available.")


In [ ]:
# ── 22.2  Retrain linear attention model to recover accuracy ─────────────────────
if 'lin_model' in dir() and 'train_loader' in dir() and 'test_loader' in dir():
    inner_model = get_inner(lin_model)
    has_attn = any(isinstance(mod, Attention) for mod in inner_model.modules())
    if not has_attn:
        print("No attention layers found in linear model. Skipping fine-tuning.")
    else:
        print("Fine-tuning linear attention model to recover accuracy...")
        from tqdm import tqdm
        
        # Enable gradients for the linear attention model
        lin_model = lin_model.to(DEVICE).train()
        for p in lin_model.parameters():
            p.requires_grad_(True)
            
        optim = torch.optim.AdamW(lin_model.parameters(), lr=1e-4)
        tr_iter = iter(train_loader)
        total_steps = 200  # fast fine-tuning
        
        pbar = tqdm(range(total_steps), desc="Fine-tuning linear model")
        running_loss = 0.0
        
        for step in pbar:
            try:
                batch = next(tr_iter)
            except StopIteration:
                tr_iter = iter(train_loader)
                batch = next(tr_iter)
                
            inputs, labels, pids = batch
            
            # Micro-batching to fit in 4GB VRAM
            micro_batch_size = 64
            num_micro_batches = max(1, len(inputs) // micro_batch_size)
            
            optim.zero_grad()
            accum_loss = 0.0
            
            for mb_idx in range(num_micro_batches):
                mb_start = mb_idx * micro_batch_size
                mb_end = mb_start + micro_batch_size
                
                mb_x = inputs[mb_start:mb_end].to(DEVICE)
                mb_y = labels[mb_start:mb_end].to(DEVICE)
                mb_pids = pids[mb_start:mb_end].to(DEVICE)
                
                mb_batch_d = {
                    "inputs": mb_x,
                    "labels": mb_y,
                    "puzzle_identifiers": mb_pids
                }
                
                sc = lin_model.initial_carry(mb_batch_d)
                sc = cast_carry_to_device(sc, DEVICE)
                for _ in range(2):
                    sc, s_out = lin_model(sc, mb_batch_d)
                    if sc.halted.all(): break
                    
                loss = F.cross_entropy(
                    s_out["logits"].reshape(-1, s_out["logits"].size(-1)),
                    mb_y.reshape(-1), ignore_index=-100
                )
                
                loss_normalized = loss / num_micro_batches
                loss_normalized.backward()
                accum_loss += loss.item()
                
            torch.nn.utils.clip_grad_norm_(lin_model.parameters(), 1.0)
            optim.step()
            
            step_loss = accum_loss / num_micro_batches
            if step == 0:
                running_loss = step_loss
            else:
                running_loss = 0.9 * running_loss + 0.1 * step_loss
                
            pbar.set_postfix(loss=f"{step_loss:.4f}", ema_loss=f"{running_loss:.4f}")
            
            if (step + 1) % 40 == 0:
                import gc
                gc.collect()
                torch.cuda.empty_cache()
                
        lin_model.eval()
        print("\nEvaluating retrained linear attention model on test set...")
        p_lin_retrained, c_lin_retrained, ms_lin_retrained, _ = evaluate_arc_per_puzzle(
            lin_model, test_loader, device=str(DEVICE), n_sup_max=1, max_batches=None
        )
        print(f"  Linear Attention (Retrained): exact={p_lin_retrained:.4f}  cell={c_lin_retrained:.4f}")
else:
    print("[SKIP] lin_model or data loaders not available.")
